# An atlas of healthy and injured cell states and niches in the human kidney

Lake et al., Nature 2023 — computational reproduction of published figures

> **Overview**: All analysis code is organized in this notebook by module.
>
> **Storage**: Data and intermediate caches reside under the project root (**RunPod** `/workspace/kidney-atlas` or **Colab** Google Drive). Paths are resolved automatically via `kidney_atlas_paths.py`.

| Module | Description |
|--------|-------------|
| 0 | Environment setup and dependency installation |
| 1 | Project initialization (1a: directories; 1b: path configuration) |
| 2 | Plotting and checkpoint utilities |
| 3 | Figure 1 reproduction (3a: data preparation; 3b: Fig. 1c) |
| 4+ | Additional figures (to be extended) |


---
## Module 0 — Environment configuration


In [ ]:
# Install dependencies (numpy<2.1 avoids numba/scanpy errors)
!pip install -q "numpy>=1.26,<2.1" scanpy>=1.10.0 anndata>=0.10.0 leidenalg>=0.10.0 python-igraph>=0.11.0 matplotlib>=3.8.0 seaborn>=0.13.0 pyarrow>=15.0.0
import numpy as np
print('numpy', np.__version__, '— if NumPy 2.5 errors persist, Runtime → Restart session, then rerun subsequent cells')


In [ ]:
# Runtime detection (Colab: mount Drive; RunPod: skip)
from pathlib import Path
import os

if Path('/content/drive/MyDrive').exists() or Path('/content/drive').exists():
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
            print('Drive mounted at /content/drive')
        else:
            print('Drive already mounted; skipping')
    except ImportError:
        print('Not Colab; skipping Drive mount')
else:
    root = os.environ.get('KIDNEY_ATLAS_ROOT', '/workspace/kidney-atlas')
    print('RunPod/local mode — project root:', root)
    print('First run: bash runpod/setup.sh')


---
## Module 1 — Project initialization

Establish the directory structure and configure path constants (auto-detected for Colab / RunPod / local execution).


### Module 1a — Directory structure

**RunPod / local**: create `data/`, `figures/`, and related directories via `mkdir`.

**Colab**: optional Google Drive API provisioning (see `setup_project_dirs_colab()` below).


In [ ]:
# Module 1a — create directories
import kidney_atlas_paths as kap

kap.setup_project_dirs()

# Colab only: run when Drive web-visible folders are required
def setup_project_dirs_colab():
    """Colab + Google Drive API (optional)."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    service = build('drive', 'v3', credentials=creds)
    print('Colab Drive API ready — extend create_folder logic as needed')

if kap.runtime_name() == 'colab':
    print('Colab: local paths created via mkdir; Drive API optional via setup_project_dirs_colab()')


### Module 1b — Path configuration

Load `kidney_atlas_paths.py` and declare output paths for Figure 1 and Figure 2.


In [ ]:
# Module 1b — path configuration (Colab / RunPod / local)
import kidney_atlas_paths as kap
kap.ensure_importable()
kap.bind_notebook_globals(globals())
kap.print_paths()


### Module 1a — Directory structure

Provision standard directories (`data/`, `figures/`, etc.) on Google Drive via the Drive API.


---
## Module 2 — Plotting and checkpoint utilities

Large objects such as `AnnData` are persisted as `.h5ad` on Drive; other Python objects use `save_checkpoint` / `load_checkpoint`.


In [ ]:
import io
import pickle

import matplotlib.pyplot as plt


def setup_scanpy():
    """Set default scanpy/matplotlib parameters (call when scanpy is available)."""
    import scanpy as sc
    sc.settings.figdir = str(FIGURES_DIR)
    sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
    plt.rcParams["figure.figsize"] = (6, 5)
    plt.rcParams["font.size"] = 10


def _get_drive_service():
    from google.colab import auth
    from googleapiclient.discovery import build
    from google.auth import default

    auth.authenticate_user()
    creds, _ = default()
    return build("drive", "v3", credentials=creds)


def _find_drive_folder_id(service, name, parent_id="root"):
    q = (
        f"name='{name}' and mimeType='application/vnd.google-apps.folder' "
        f"and trashed=false and '{parent_id}' in parents"
    )
    files = service.files().list(q=q, fields="files(id)").execute().get("files", [])
    return files[0]["id"] if files else None


def save_figure_to_drive(fig, filename: str, dpi: int = 300, **savefig_kw) -> str:
    """Upload figure via Drive API for cloud/web visibility."""
    from googleapiclient.http import MediaIoBaseUpload

    service = _get_drive_service()
    project_id = _find_drive_folder_id(service, "human-kidney-scrna-atlas-reproduction")
    if not project_id:
        raise FileNotFoundError("Project folder not found; run Module 1a setup_project_dirs() first")
    figures_id = _find_drive_folder_id(service, "figures", project_id)
    if not figures_id:
        raise FileNotFoundError("figures/ folder not found; run Module 1a first")

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", facecolor="white", **savefig_kw)
    buf.seek(0)

    q = f"name='{filename}' and '{figures_id}' in parents and trashed=false"
    existing = service.files().list(q=q, fields="files(id)").execute().get("files", [])
    media = MediaIoBaseUpload(buf, mimetype="image/png", resumable=True)
    if existing:
        fid = service.files().update(fileId=existing[0]["id"], media_body=media).execute()["id"]
    else:
        fid = service.files().create(
            body={"name": filename, "parents": [figures_id]},
            media_body=media,
            fields="id",
        ).execute()["id"]
    url = f"https://drive.google.com/file/d/{fid}/view"
    print(f"Uploaded to Drive: {filename}")
    print(f"  {url}")
    return fid


def save_current_figure(name: str, dpi: int = 300) -> str:
    fid = save_figure_to_drive(plt.gcf(), name, dpi=dpi)
    plt.close()
    return fid


def save_checkpoint(name: str, obj):
    path = CHECKPOINT_DIR / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print("Checkpoint saved:", path)
    return path


def load_checkpoint(name: str, default=None):
    path = CHECKPOINT_DIR / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            obj = pickle.load(f)
        print("Checkpoint loaded:", path)
        return obj
    return default


print("Module 2 utilities loaded (Fig. 2g requires save_figure_to_drive only)")


---
## Module 3 — Figure 1 reproduction

Reproduce Figure 1c: joint UMAP of integrated RNA data with cell-type annotations.


### Module 3a — Data preparation

Load **UMAP coordinates and annotation columns only** via `h5py` (without loading the ~3 GB expression matrix). When data reside on Drive, a one-time copy to Colab local storage precedes reading. Designed for 12 GB RAM environments.


In [ ]:
import gc
import shutil
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scanpy as sc


def find_cellxgene_h5ad(directory: Path) -> Path:
    files = sorted(directory.glob("*.h5ad"))
    if not files:
        raise FileNotFoundError(f"h5ad not found: {directory}")
    if len(files) > 1:
        print("Multiple h5ad files found; using:", files[0].name)
    return files[0]


def _read_obs_categorical(h5f: h5py.File, col: str) -> np.ndarray:
    grp = h5f["obs"][col]
    categories = grp["categories"][:].astype(str)
    codes = grp["codes"][:]
    return categories[codes]


def _local_h5ad_copy(src_path: Path) -> Path:
    src_path = Path(src_path)
    if not str(src_path).startswith("/content/drive"):
        return src_path
    local_path = Path("/content") / src_path.name
    if local_path.exists() and local_path.stat().st_size == src_path.stat().st_size:
        print("Using Colab local copy:", local_path)
        return local_path
    print("Copying to Colab local disk (one-time, ~3 GB disk, no RAM)...")
    shutil.copy2(src_path, local_path)
    print("Copy complete:", local_path)
    return local_path


def prepare_fig1_adata(src_path: Path, cache_path: Path) -> sc.AnnData:
    if cache_path.exists():
        print("Loading from lightweight cache:", cache_path)
        adata = sc.read_h5ad(cache_path)
        need = [FIG1C_COLOR_KEY, "subclass.l1", "suspension_type", "assay"]
        if all(c in adata.obs.columns for c in need):
            return adata
        print("Cache missing subclass.l1; re-extracting...")

    keep_obs = [FIG1C_COLOR_KEY, "subclass.l1", "suspension_type", "assay"]
    read_path = _local_h5ad_copy(src_path)
    print("h5py column-wise read:", read_path)

    with h5py.File(read_path, "r") as h5f:
        if "obsm/X_umap" not in h5f:
            raise ValueError("Missing obsm/X_umap")
        for col in keep_obs:
            if col not in h5f["obs"]:
                raise KeyError(f"Missing obs/{col}")
        umap = h5f["obsm/X_umap"][:].astype(np.float32)
        obs = pd.DataFrame({col: _read_obs_categorical(h5f, col) for col in keep_obs})
    gc.collect()

    adata = sc.AnnData(obs=obs, obsm={"X_umap": umap})
    adata.uns["source_h5ad"] = str(src_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(cache_path)

    print("Cell count:", adata.n_obs)
    print(f"{FIG1C_COLOR_KEY} unique types:", adata.obs[FIG1C_COLOR_KEY].nunique())
    print("Saved lightweight cache (~16 MB):", cache_path)
    return adata


for stale in [PROCESSED_DATA_DIR / "fig1_integrated.h5ad"]:
    if stale.exists():
        stale.unlink()
        print("Removed stale full cache:", stale)

if not CELLXGENE_DIR.exists():
    raise FileNotFoundError(f"Directory not found: {CELLXGENE_DIR}")

src_h5ad = find_cellxgene_h5ad(CELLXGENE_DIR)
adata = prepare_fig1_adata(src_h5ad, CACHED_FIG1)
print(adata)


### Module 3b — Figure 1c (annotated UMAP)

Pastel color palette with direct on-plot labels for major cell types (no separate legend panel), consistent with the published layout.


In [ ]:
import colorsys

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

adata = sc.read_h5ad(CACHED_FIG1)
umap = adata.obsm["X_umap"]

# ── Pastel palette: hue by subclass.l1; shade variants per subclass.l3 ──
L1_MACARON = {
    "PT": "#FF7E79",
    "TAL": "#66E0A3",
    "IMM": "#FF8EC8",
    "EC": "#5EC8FF",
    "PC": "#B2FF59",
    "IC": "#5DFFD5",
    "FIB": "#D4A5FF",
    "CNT": "#8AE5A8",
    "DCT": "#D4F067",
    "DTL": "#FFB870",
    "VSM/P": "#FF99C2",
    "ATL": "#FFE566",
    "POD": "#FF6EB4",
    "PEC": "#E0B0FF",
    "PapE": "#64FFDA",
    "NEUR": "#7FD4FF",
}


def _hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i : i + 2], 16) / 255 for i in (0, 2, 4))


def _rgb_to_hex(r, g, b):
    return "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))


def macaron_variants(base_hex: str, n: int) -> list:
    if n <= 1:
        return [base_hex]
    r, g, b = _hex_to_rgb(base_hex)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    colors = []
    for i in range(n):
        lightness = 0.58 + 0.18 * i / (n - 1)
        sat = min(0.92, max(0.65, s))
        rr, gg, bb = colorsys.hls_to_rgb(h, lightness, sat)
        colors.append(_rgb_to_hex(rr, gg, bb))
    return colors


def build_l3_palette(obs) -> dict:
    palette = {}
    for l1, sub in obs.groupby("subclass.l1")["subclass.l3"].unique().items():
        base = L1_MACARON.get(l1, "#FFB6C1")
        for l3, color in zip(sorted(sub), macaron_variants(base, len(sub))):
            palette[l3] = color
    return palette


PALETTE_L3 = build_l3_palette(adata.obs)

FIG1C_LABELS = {
    "IC": "IC", "PC": "PC", "CNT": "CNT", "DCT": "DCT", "PapE": "PapE",
    "POD": "POD", "VSM/P": "VSM/P", "FIB": "FIB", "PT": "PT", "TAL": "TAL",
    "ATL": "ATL", "DTL": "DTL", "IMM": "Immune\ncells", "EC": "Endothelial\ncells",
    "NEUR": "Neural\ncells",
}
EPITHELIAL_L1 = {"PT", "TAL", "IC", "PC", "CNT", "DCT", "DTL", "ATL", "POD", "PEC", "PapE"}


def plot_fig1c_labeled(ax):
    for l3, color in PALETTE_L3.items():
        mask = adata.obs[FIG1C_COLOR_KEY].values == l3
        ax.scatter(umap[mask, 0], umap[mask, 1], c=color, s=1.2, linewidths=0, rasterized=True)
    for l1, label in FIG1C_LABELS.items():
        mask = adata.obs["subclass.l1"].values == l1
        if mask.sum() < 30:
            continue
        cx, cy = umap[mask].mean(axis=0)
        ax.text(cx, cy, label, fontsize=8, ha="center", va="center", color="#333333", clip_on=False)
    epi_mask = adata.obs["subclass.l1"].isin(EPITHELIAL_L1).values
    if epi_mask.sum() > 0:
        cx, cy = umap[epi_mask].mean(axis=0)
        ax.text(cx + 1.5, cy + 1.0, "Epithelial\ncells", fontsize=8, ha="center", va="center", color="#555555")
    stromal_mask = adata.obs["subclass.l1"].isin({"FIB", "VSM/P"}).values
    if stromal_mask.sum() > 0:
        cx, cy = umap[stromal_mask].mean(axis=0)
        ax.text(cx - 2.0, cy - 1.5, "Stromal\ncells", fontsize=8, ha="center", va="center", color="#555555")
    ax.set_title("Figure 1c — integrated UMAP", fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


fig, ax = plt.subplots(figsize=(7, 6))
plot_fig1c_labeled(ax)
plt.tight_layout()
FIG1C_OUTPUT = FIGURES_DIR / "figure1c_labeled.png"
FIG1C_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG1C_OUTPUT, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", FIG1C_OUTPUT)


### Module 3c — Figure 1c modality-specific UMAP

The manuscript displays three assays separately — **snCv3**, **SNARE2**, and **scCv3** (~one third of cells each), with modality-specific coverage bias on the UMAP.

This CELLxGENE release **lacks SNARE2 labels**; `suspension_type` is used as a proxy:
- `nucleus` → snCv3 (200,338 cells; **66%**)
- `cell` → scCv3 (104,314 cells; **34%**)

Consequently, the snCv3 panel appears **more densely populated** than in the manuscript: nuclei are well represented across most cell types (~41–98%). This reflects dataset composition, not a UMAP error. Point size and transparency are reduced below to better approximate the published appearance.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

adata = sc.read_h5ad(CACHED_FIG1)
umap = adata.obsm["X_umap"]

MODALITY_PANELS = [
    ("nucleus", "snCv3", "#4A90E2", "nuclei"),
    ("cell", "scCv3", "#1ABC9C", "cells"),
]

# Manuscript uses very small, semi-transparent points; s=0.55 yields solid blocks at ~300k points
BG_STYLE = dict(c="#D0D0D0", s=0.06, alpha=0.55, linewidths=0, rasterized=True, zorder=1)
FG_STYLE = dict(s=0.04, alpha=0.85, linewidths=0, rasterized=True, zorder=2)


def plot_modality_panel(ax, modality_value, title, color, unit):
    mask = adata.obs["suspension_type"].values == modality_value
    bg = ~mask
    ax.scatter(umap[bg, 0], umap[bg, 1], **BG_STYLE)
    ax.scatter(umap[mask, 0], umap[mask, 1], c=color, **FG_STYLE)
    ax.set_title(title, fontsize=13, color=color, fontweight="bold", loc="left")
    ax.text(0.02, 0.02, f"{mask.sum():,} {unit}", transform=ax.transAxes, fontsize=10, va="bottom")
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), dpi=120)
for ax, (modality, title, color, unit) in zip(axes, MODALITY_PANELS):
    plot_modality_panel(ax, modality, title, color, unit)

plt.tight_layout()
FIG1C_MODALITIES_OUTPUT = FIGURES_DIR / "figure1c_modalities.png"
FIG1C_MODALITIES_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG1C_MODALITIES_OUTPUT, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", FIG1C_MODALITIES_OUTPUT)

# Quick check: nucleus fraction per cell type (explains denser snCv3 panel)
frac = adata.obs.groupby("subclass.l1")["suspension_type"].apply(lambda s: (s == "nucleus").mean())
print("Nucleus fraction (subclass.l1):\n", frac.sort_values().to_string())


---
## Module 4 — Figure 2b reproduction

Manuscript Fig. 2b: snCv3 `subclass.l3` UMAP with **anatomical region** and **altered-state** inset panels.

> CELLxGENE provides no explicit snCv3 label; `suspension_type == "nucleus"` is used as a proxy (200,338 nuclei).
> The UMAP is a subset projection of the integrated embedding (the manuscript uses snCv3-specific dimensionality reduction; topology may differ slightly).


### Module 4a — snCv3 data preparation

Lightweight `h5py` extraction of the nucleus subset with `subclass.l3`, `class`, `region.l2`, and `state.l2`.


In [ ]:
import gc
import shutil
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scanpy as sc


def _read_obs_categorical(h5f: h5py.File, col: str) -> np.ndarray:
    grp = h5f["obs"][col]
    categories = grp["categories"][:].astype(str)
    codes = grp["codes"][:]
    return categories[codes]


def _local_h5ad_copy(src_path: Path) -> Path:
    src_path = Path(src_path)
    if not str(src_path).startswith("/content/drive"):
        return src_path
    local_path = Path("/content") / src_path.name
    if local_path.exists() and local_path.stat().st_size == src_path.stat().st_size:
        return local_path
    print("Copying to Colab local disk...")
    shutil.copy2(src_path, local_path)
    return local_path


def prepare_fig2b_adata(src_path: Path, cache_path: Path) -> sc.AnnData:
    need_cols = [FIG2B_COLOR_KEY, "subclass.l1", "class", "region.l2", "state.l2", "suspension_type"]
    if cache_path.exists():
        adata = sc.read_h5ad(cache_path)
        if all(c in adata.obs.columns for c in need_cols):
            print("Loading from lightweight cache:", cache_path)
            return adata
        print("Cache columns incomplete; re-extracting...")

    read_path = _local_h5ad_copy(src_path)
    print("h5py reading snCv3 subset:", read_path)
    with h5py.File(read_path, "r") as h5f:
        umap = h5f["obsm/X_umap"][:].astype(np.float32)
        obs = pd.DataFrame({col: _read_obs_categorical(h5f, col) for col in need_cols})
    gc.collect()

    sn_mask = obs["suspension_type"].values == "nucleus"
    adata = sc.AnnData(
        obs=obs.loc[sn_mask].reset_index(drop=True),
        obsm={"X_umap": umap[sn_mask]},
    )
    adata.uns["source_h5ad"] = str(src_path)
    adata.uns["note"] = "snCv3 proxy: suspension_type == nucleus"

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(cache_path)
    print("Cell count:", adata.n_obs)
    print(f"{FIG2B_COLOR_KEY} unique types:", adata.obs[FIG2B_COLOR_KEY].nunique())
    print("Saved:", cache_path)
    return adata


src_h5ad = sorted(CELLXGENE_DIR.glob("*.h5ad"))[0]
adata_sn = prepare_fig2b_adata(src_h5ad, CACHED_FIG2B)
print(adata_sn)


### Module 4b — Figure 2b plotting

Three figures are **saved separately** (square aspect ratio to avoid distortion):
- `figure2b_sncv3_celltypes.png` — main panel
- `figure2b_sncv3_altered_states.png` — altered states
- `figure2b_sncv3_regions.png` — anatomical regions


In [ ]:
import colorsys

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from matplotlib.lines import Line2D

adata = sc.read_h5ad(CACHED_FIG2B)
umap = adata.obsm["X_umap"]

PAD = 1.0
XLIM = (umap[:, 0].min() - PAD, umap[:, 0].max() + PAD)
YLIM = (umap[:, 1].min() - PAD, umap[:, 1].max() + PAD)

L1_MACARON = {
    "PT": "#FF9A76", "TAL": "#7DDE92", "IMM": "#FF7EB9", "EC": "#5EB7FF",
    "PC": "#B8F26A", "IC": "#5EFFE0", "FIB": "#D9A8FF", "CNT": "#9AE8A8",
    "DCT": "#E0F070", "DTL": "#FFC266", "VSM/P": "#FFAAC8", "ATL": "#FFEE70",
    "POD": "#FF7EC8", "PEC": "#E8C0FF", "PapE": "#70FFE0", "NEUR": "#8FD8FF",
}


def _hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i : i + 2], 16) / 255 for i in (0, 2, 4))


def _rgb_to_hex(r, g, b):
    return "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))


def macaron_variants(base_hex, n):
    if n <= 1:
        return [base_hex]
    r, g, b = _hex_to_rgb(base_hex)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    out = []
    for i in range(n):
        lightness = 0.55 + 0.2 * i / (n - 1)
        sat = min(0.9, max(0.62, s))
        rr, gg, bb = colorsys.hls_to_rgb(h, lightness, sat)
        out.append(_rgb_to_hex(rr, gg, bb))
    return out


def build_l3_palette(obs):
    palette = {}
    for l1, sub in obs.groupby("subclass.l1")[FIG2B_COLOR_KEY].unique().items():
        base = L1_MACARON.get(l1, "#FFB6C1")
        for l3, color in zip(sorted(sub), macaron_variants(base, len(sub))):
            palette[l3] = color
    for l3 in obs[FIG2B_COLOR_KEY].unique():
        palette.setdefault(l3, "#CCCCCC")
    return palette


PALETTE_L3 = build_l3_palette(adata.obs)

CLASS_LABELS = {
    "endothelial cells": "Endothelial\ncells",
    "immune cells": "Immune\ncells",
    "stroma cells": "Stromal\ncells",
    "epithelial cells": "Epithelial\ncells",
}
CLASS_OFFSETS = {
    "endothelial cells": (-3.8, 0.4),
    "immune cells": (0.8, 3.2),
    "stroma cells": (-3.2, -2.4),
    "epithelial cells": (5.2, 1.8),
}
SEED_OFFSETS = {
    "CCD-IC-A": (-1.0, 1.6), "OMCD-IC-A": (1.2, 1.4), "CNT-IC-A": (0.0, -1.3),
    "tPC-IC": (1.4, -0.6), "IC-B": (-1.4, -1.0), "MDC": (1.6, 0.4),
    "MAC-M2": (-1.5, 0.3), "ncMON": (0.0, 1.5), "T": (-0.8, 1.0),
    "NKC/T": (1.0, 1.0), "B": (-1.0, -0.8), "EC-PTC": (0.0, 1.4),
    "dEC-PTC": (1.2, 0.8), "MYOF": (1.2, 0.0), "VSMC/P": (-1.0, 0.6),
    "MC": (0.8, -0.8), "dC-TAL": (-1.0, 0.6), "dM-TAL": (1.0, 0.6),
    "OMCD-PC": (-1.0, 0.5), "CCD-PC": (1.0, 0.5), "dDTL3": (-0.8, 0.6),
    "aATL": (0.8, 0.6), "cycPT": (-1.2, 0.5),
}

STATE_STYLE = {
    "reference": ("Ref", "#E6E6E6"), "adaptive - epi": ("aEpi", "#B8F55A"),
    "adaptive - str": ("aStr", "#FFD24A"), "cycling": ("Cyc", "#5EB8FF"),
    "transitioning": ("Trans", "#4DFFE8"), "degenerative": ("Degen", "#FF8FAB"),
}
REGION_STYLE = {
    "C": ("Cortex", "#7EE787"), "C-M": ("Cortex/Med.", "#6EC1FF"),
    "M": ("Medulla", "#D896FF"), "P": ("Papilla", "#FF7EC8"),
}


def style_umap_ax(ax):
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


def scatter_umap(ax, mask, color, size=0.35, alpha=0.9, zorder=2):
    ax.scatter(umap[mask, 0], umap[mask, 1], c=color, s=size, linewidths=0, alpha=alpha, rasterized=True, zorder=zorder)


def add_legend(ax, items, y0=0.80):
    handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=6, linestyle="") for _, c in items]
    labels = [lb for lb, _ in items]
    ax.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.02, y0), fontsize=8, frameon=False, labelspacing=0.55)


def _label_radius(text, bold=False):
    return 1.9 if bold else 0.55 + 0.045 * len(text.replace("\n", ""))


def repel_label_positions(positions, radii, max_iter=500):
    pos = positions.astype(float).copy()
    n = len(pos)
    for _ in range(max_iter):
        moved = False
        for i in range(n):
            for j in range(i + 1, n):
                delta = pos[i] - pos[j]
                dist = float(np.hypot(delta[0], delta[1]))
                need = radii[i] + radii[j]
                if dist < need:
                    if dist < 1e-4:
                        delta = np.array([0.05, 0.03]); dist = float(np.hypot(delta[0], delta[1]))
                    shift = delta * ((need - dist) / dist * 0.55)
                    pos[i] += shift; pos[j] -= shift; moved = True
        if not moved:
            break
    return pos


def place_umap_labels(ax):
    entries = []
    for cls, label in CLASS_LABELS.items():
        mask = adata.obs["class"].values == cls
        if mask.sum() < 50: continue
        cx, cy = umap[mask].mean(axis=0)
        ox, oy = CLASS_OFFSETS.get(cls, (0.0, 0.0))
        entries.append({"text": label, "anchor": (cx, cy), "start": (cx+ox, cy+oy), "fontsize": 9, "bold": True})
    for l3, cnt in adata.obs[FIG2B_COLOR_KEY].value_counts().items():
        if cnt < 200: continue
        mask = adata.obs[FIG2B_COLOR_KEY].values == l3
        cx, cy = umap[mask].mean(axis=0)
        ox, oy = SEED_OFFSETS.get(l3, (0.0, 0.0))
        entries.append({"text": l3, "anchor": (cx, cy), "start": (cx+ox, cy+oy), "fontsize": 7.5, "bold": False})
    if not entries: return
    starts = np.array([e["start"] for e in entries], dtype=float)
    radii = np.array([_label_radius(e["text"], e["bold"]) for e in entries], dtype=float)
    final = repel_label_positions(starts, radii)
    margin = 0.3
    final[:, 0] = np.clip(final[:, 0], XLIM[0]+margin, XLIM[1]-margin)
    final[:, 1] = np.clip(final[:, 1], YLIM[0]+margin, YLIM[1]-margin)
    for e, (lx, ly) in zip(entries, final):
        ax0, ay0 = e["anchor"]
        if float(np.hypot(lx-ax0, ly-ay0)) > 0.55:
            ax.plot([ax0, lx], [ay0, ly], color="#999999", lw=0.35, alpha=0.7, zorder=8)
        ax.text(lx, ly, e["text"], fontsize=e["fontsize"], ha="center", va="center",
                color="#222222" if e["bold"] else "#333333", fontweight="bold" if e["bold"] else "normal",
                zorder=10, clip_on=False)


def plot_main_celltypes(ax):
    for l3, color in PALETTE_L3.items():
        mask = adata.obs[FIG2B_COLOR_KEY].values == l3
        if mask.sum() == 0: continue
        scatter_umap(ax, mask, color, size=0.45, alpha=0.88)
    place_umap_labels(ax)
    ax.set_title("snCv3 — subclass.l3", fontsize=12, loc="left")


def plot_altered_states(ax):
    ref_mask = adata.obs["state.l2"].values == "reference"
    scatter_umap(ax, ref_mask, STATE_STYLE["reference"][1], size=0.28, alpha=0.5, zorder=1)
    draw_order = ["adaptive - epi", "adaptive - str", "cycling", "transitioning", "degenerative"]
    for key in draw_order:
        _, color = STATE_STYLE[key]
        mask = adata.obs["state.l2"].values == key
        if mask.sum() == 0: continue
        scatter_umap(ax, mask, color, size=0.38, alpha=0.92, zorder=2)
    ax.set_title("Altered states", fontsize=12)
    add_legend(ax, [STATE_STYLE[k] for k in ["reference"] + draw_order], y0=0.80)


def plot_regions(ax):
    for key, (_, color) in REGION_STYLE.items():
        mask = adata.obs["region.l2"].values == key
        if mask.sum() == 0: continue
        scatter_umap(ax, mask, color, size=0.38, alpha=0.92)
    ax.set_title("Regions", fontsize=12)
    add_legend(ax, [REGION_STYLE[k] for k in REGION_STYLE], y0=0.80)


def save_panel(plot_fn, path, figsize=(8.5, 8.5)):
    fig, ax = plt.subplots(figsize=figsize, dpi=120)
    plot_fn(ax); style_umap_ax(ax)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0.15); plt.show(); plt.close(fig)
    print("Saved:", path)


save_panel(plot_main_celltypes, FIG2B_MAIN_OUTPUT)
save_panel(plot_altered_states, FIG2B_STATES_OUTPUT)
save_panel(plot_regions, FIG2B_REGIONS_OUTPUT)


---
## Module 5 — Figure 2c reproduction

Manuscript Fig. 2c triptych:
1. **Left**: Slide-seq cell-type frequency heatmap along the cortico-medullary axis (Cx / OM / O-IM / IM) for three donors
2. **Center**: spatial transition ATL → M-TAL on a representative puck (`subclass.l2`)
3. **Right**: scaled spatial expression of marker genes *SLC12A1*, *SLC14A1*, and *SH3GL3*

> Data: `data/raw/slideseq/GSE183274/pucks/*.rds.gz`
>
> The first run of **5a** installs R + Seurat (~5–10 min); extracted results are cached under `data/processed/`.


In [ ]:
# Figure 2c paths and constants
RAW_DATA_DIR = DATA_DIR / "raw"
SLIDESEQ_DIR = RAW_DATA_DIR / "slideseq" / "GSE183274"
SLIDESEQ_PUCKS = SLIDESEQ_DIR / "pucks"
SLIDESEQ_MANIFEST = SLIDESEQ_DIR / "metadata" / "sample_manifest.tsv"
CACHED_FIG2C_BEADS = PROCESSED_DATA_DIR / "fig2c_all_beads.parquet"
CACHED_FIG2C_SPATIAL = PROCESSED_DATA_DIR / "fig2c_spatial_crop.parquet"
CACHED_FIG2C_HEATMAP = PROCESSED_DATA_DIR / "fig2c_heatmap_matrix.csv"
FIG2C_HEATMAP_OUTPUT = FIGURES_DIR / "figure2c_heatmap.png"
FIG2C_SPATIAL_TYPES_OUTPUT = FIGURES_DIR / "figure2c_spatial_celltypes.png"
FIG2C_MARKERS_OUTPUT = FIGURES_DIR / "figure2c_spatial_markers.png"
FIG2C_COMBINED_OUTPUT = FIGURES_DIR / "figure2c_combined.png"
FIG2C_DONORS = ["201229", "210412", "210113"]
FIG2C_SPATIAL_PUCK = "Puck_210113_23"
FIG2C_MARKERS = ["SLC12A1", "SLC14A1", "SH3GL3"]
FIG2C_AXIS_REGIONS = ["Cx", "OM", "O-IM", "IM"]
FIG2C_TYPE_COL = "subclass.l2"
FIG2C_PAPER_CELL_ORDER = [
    "PEC", "POD", "MC", "EC-GC", "REN", "EC-AEA", "VSMC",
    "PT-S1", "PT-S2", "PT-S3", "aPT", "EC-PTC", "EC-LYM", "FIB", "MyoF", "aFIB",
    "C-TAL", "aTAL1", "MD", "DCT", "CNT", "C-PC", "C-IC-A", "IC-B",
    "DTL1", "DTL2", "EC-DVR", "VSMC/P", "M-TAL", "M-PC", "M-IC-A",
    "EC-AVR", "DTL3", "ATL", "IMCD", "M-FIB",
]
FIG2C_CELL_TYPE_ALIASES = {"MYOF": "MyoF"}
# Manuscript Methods: Fig. 2c uses beads with subclass.l2 max weight ≥ 50%
FIG2C_MIN_WEIGHT_L2 = 50.0
print("Pucks:", len(list(SLIDESEQ_PUCKS.glob("*.rds.gz"))), "manifest:", SLIDESEQ_MANIFEST.exists())


### Module 5a — Slide-seq metadata extraction (Python / rdata)

GEO `*.rds.gz` files are **double-gzip**-compressed **Giotto** objects (not Seurat). Parsed with `rdata`; no R runtime required.


In [ ]:
# Module 5a — Python extraction (double-gzip Giotto RDS; no R/Seurat required)
import gzip
import io
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

!pip install -q rdata pyarrow

import rdata
import scipy.sparse as sparse

warnings.filterwarnings("ignore", category=UserWarning, module="rdata")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


def read_giotto_puck(path: Path):
    """GEO KPMP *.rds.gz are double-gzip Giotto objects."""
    raw = gzip.decompress(gzip.decompress(path.read_bytes()))
    parsed = rdata.parser.parse_file(io.BytesIO(raw))
    return rdata.conversion.convert(parsed)


def _as_dataframe(obj) -> pd.DataFrame:
    if isinstance(obj, pd.DataFrame):
        out = obj.copy()
    else:
        out = pd.DataFrame(obj)
    out.columns = [str(c) for c in out.columns]
    return out


def _normalize_cell_type(series: pd.Series) -> pd.Series:
    return series.astype(str).str.replace(r"^l[12]\.", "", regex=True)


def _pick_type_col(df: pd.DataFrame) -> str:
    for c in ["subclass.l2", "maxCelltype.l2", "subclass_l2", "subclass.l3", "predicted.id"]:
        if c in df.columns:
            return c
    for c in df.columns:
        cl = str(c).lower()
        if "subclass" in cl and "l2" in cl:
            return c
        if "maxcelltype" in cl and "l2" in cl:
            return c
    raise KeyError(f"No cell type column: {list(df.columns)[:30]}")


def _coord_cols(df: pd.DataFrame) -> tuple[str, str]:
    for x, y in [("sdimx", "sdimy"), ("x", "y"), ("imagecol", "imagerow"), ("row", "col")]:
        if x in df.columns and y in df.columns:
            return x, y
    raise KeyError(f"No coordinate columns in {list(df.columns)[:20]}")


def _expr_triplet(gobj):
    for attr in ("raw_exprs", "norm_expr", "expression"):
        if not hasattr(gobj, attr):
            continue
        node = getattr(gobj, attr)
        if hasattr(node, "i") and hasattr(node, "j") and hasattr(node, "x") and hasattr(node, "Dim"):
            genes = [str(g) for g in node.Dimnames[0]]
            return node, genes
    return None, None


def _raw_gene(gobj, gene: str, n_cells: int) -> np.ndarray | None:
    """Raw counts (no normalization; aligned with manuscript gene color scale)."""
    triplet, genes = _expr_triplet(gobj)
    if triplet is None:
        return None
    upper = [g.upper() for g in genes]
    if gene.upper() not in upper:
        return None
    gi = upper.index(gene.upper())
    mat = sparse.coo_matrix((triplet.x, (triplet.i, triplet.j)), shape=tuple(triplet.Dim)).tocsr()
    vals = np.asarray(mat.getrow(gi).toarray()).ravel()
    return vals if len(vals) == n_cells else None


def extract_puck(path: Path, puck: str, region: str) -> pd.DataFrame:
    gobj = read_giotto_puck(path)
    meta = _as_dataframe(gobj.cell_metadata)
    locs = _as_dataframe(gobj.spatial_locs)
    id_col = "cell_ID" if "cell_ID" in meta.columns else meta.columns[0]
    df = meta.merge(locs, on=id_col, how="inner") if id_col in locs.columns else pd.concat(
        [meta.reset_index(drop=True), locs.reset_index(drop=True)], axis=1
    )
    type_col = _pick_type_col(df)
    xcol, ycol = _coord_cols(df)
    donor = puck.split("_")[1] if puck.startswith("Puck_") else puck
    out = pd.DataFrame(
        {
            "puck": puck,
            "donor": donor,
            "puck_region": region,
            "x": pd.to_numeric(df[xcol], errors="coerce"),
            "y": pd.to_numeric(df[ycol], errors="coerce"),
            "cell_type": _normalize_cell_type(df[type_col]),
            "max_weight_l2": pd.to_numeric(df.get("maxWeight.l2"), errors="coerce"),
        }
    )
    return out.dropna(subset=["x", "y"])


FIG2C_WEIGHT_TYPES = ["M-TAL", "EC-DVR", "ATL"]


def extract_spatial_crop(path: Path, puck: str) -> pd.DataFrame:
    gobj = read_giotto_puck(path)
    meta = _as_dataframe(gobj.cell_metadata)
    locs = _as_dataframe(gobj.spatial_locs)
    id_col = "cell_ID" if "cell_ID" in meta.columns else meta.columns[0]
    df = meta.merge(locs, on=id_col, how="inner") if id_col in locs.columns else pd.concat(
        [meta.reset_index(drop=True), locs.reset_index(drop=True)], axis=1
    )
    type_col = _pick_type_col(df)
    xcol, ycol = _coord_cols(df)
    n = len(df)
    spatial_df = pd.DataFrame(
        {
            "x": pd.to_numeric(df[xcol], errors="coerce"),
            "y": pd.to_numeric(df[ycol], errors="coerce"),
            "cell_type": _normalize_cell_type(df[type_col]),
            "max_weight_l2": pd.to_numeric(df.get("maxWeight.l2"), errors="coerce"),
        }
    )
    # Full per-type RCTD weights (l2.<type>, 0–100)
    for ct in FIG2C_WEIGHT_TYPES:
        col = f"l2.{ct}"
        spatial_df[f"w_{ct}"] = pd.to_numeric(df.get(col), errors="coerce") if col in df.columns else np.nan
    # Marker genes: raw counts
    for gene in FIG2C_MARKERS:
        vals = _raw_gene(gobj, gene, n)
        spatial_df[gene] = vals if vals is not None else np.nan
        if vals is None:
            print(f"warning: could not extract {gene} from {puck}")

    spatial_df = spatial_df.dropna(subset=["x", "y"])
    # Return full puck; rotation + axis-aligned crop in 5c (avoids diamond artifact from pre-rotation crop)
    return spatial_df


def _cache_ok(path: Path) -> bool:
    return path.exists() and path.stat().st_size >= 1000

for p in (CACHED_FIG2C_BEADS, CACHED_FIG2C_SPATIAL):
    if p.exists() and p.stat().st_size < 1000:
        p.unlink()
        print("Removing incomplete cache:", p)

manifest = pd.read_csv(SLIDESEQ_MANIFEST, sep="\t")

def _beads_cache_ok(path: Path) -> bool:
    if not _cache_ok(path):
        return False
    try:
        pd.read_parquet(path, columns=["max_weight_l2"])
        return True
    except Exception:
        return False

if _beads_cache_ok(CACHED_FIG2C_BEADS):
    print("Beads cache exists; skipping 67-puck extraction:", CACHED_FIG2C_BEADS)
else:
    manifest["donor"] = manifest["puck"].str.extract(r"Puck_(\d+)_")[0]
    print(manifest.groupby("donor").size().sort_index())
    rows = []
    for _, row in manifest.iterrows():
        fpath = SLIDESEQ_PUCKS / row["rds_file"]
        if not fpath.exists():
            print("missing", fpath.name)
            continue
        print("Reading", row["puck"])
        rows.append(extract_puck(fpath, row["puck"], row["region"]))
    beads = pd.concat(rows, ignore_index=True)
    beads.to_parquet(CACHED_FIG2C_BEADS, index=False)
    print("beads:", beads.shape, "->", CACHED_FIG2C_BEADS)

# Single representative puck; recompute spatial each run (~10 s) to match FIG2C_SPATIAL_PUCK
spatial_rds = SLIDESEQ_PUCKS / manifest.loc[
    manifest["puck"] == FIG2C_SPATIAL_PUCK, "rds_file"
].iloc[0]
spatial_crop = extract_spatial_crop(spatial_rds, FIG2C_SPATIAL_PUCK)
spatial_crop.to_parquet(CACHED_FIG2C_SPATIAL, index=False)
print("spatial crop:", spatial_crop.shape, "->", CACHED_FIG2C_SPATIAL)

print("Module 5a complete")


### Module 5b — Cortico-medullary axis heatmap matrix


In [ ]:

# ── Fig. 2c constants fallback (if path cell was not run) ──
if "FIG2C_MIN_WEIGHT_L2" not in globals():
    FIG2C_DONORS = ["201229", "210412", "210113"]
    FIG2C_SPATIAL_PUCK = "Puck_210113_23"
    FIG2C_MARKERS = ["SLC12A1", "SLC14A1", "SH3GL3"]
    FIG2C_AXIS_REGIONS = ["Cx", "OM", "O-IM", "IM"]
    FIG2C_CELL_TYPE_ALIASES = {"MYOF": "MyoF"}
    FIG2C_MIN_WEIGHT_L2 = 50.0
    FIG2C_PAPER_CELL_ORDER = [
        "PEC", "POD", "MC", "EC-GC", "REN", "EC-AEA", "VSMC",
        "PT-S1", "PT-S2", "PT-S3", "aPT", "EC-PTC", "EC-LYM", "FIB", "MyoF", "aFIB",
        "C-TAL", "aTAL1", "MD", "DCT", "CNT", "C-PC", "C-IC-A", "IC-B",
        "DTL1", "DTL2", "EC-DVR", "VSMC/P", "M-TAL", "M-PC", "M-IC-A",
        "EC-AVR", "DTL3", "ATL", "IMCD", "M-FIB",
    ]
    CACHED_FIG2C_BEADS = PROCESSED_DATA_DIR / "fig2c_all_beads.parquet"
    CACHED_FIG2C_SPATIAL = PROCESSED_DATA_DIR / "fig2c_spatial_crop.parquet"
    CACHED_FIG2C_HEATMAP = PROCESSED_DATA_DIR / "fig2c_heatmap_matrix.csv"
    FIG2C_HEATMAP_OUTPUT = FIGURES_DIR / "figure2c_heatmap.png"
    FIG2C_SPATIAL_TYPES_OUTPUT = FIGURES_DIR / "figure2c_spatial_celltypes.png"
    FIG2C_MARKERS_OUTPUT = FIGURES_DIR / "figure2c_spatial_markers.png"
    FIG2C_COMBINED_OUTPUT = FIGURES_DIR / "figure2c_combined.png"
    print("Auto-filled Fig. 2c constants (path cell still recommended)")

import numpy as np
import pandas as pd

# Medullary depth anchor types: outer vs inner medulla/papilla (for puck ordering)
FIG2C_OUTER_MED = ["M-TAL", "C-TAL", "DTL1"]
FIG2C_INNER_MED = ["IMCD", "ATL", "DTL3", "M-FIB", "M-IC-A"]

beads = pd.read_parquet(CACHED_FIG2C_BEADS)
beads = beads[beads["donor"].astype(str).isin(FIG2C_DONORS)].copy()
beads = beads[beads["cell_type"].notna() & (beads["cell_type"] != "")]
beads["cell_type"] = beads["cell_type"].replace(FIG2C_CELL_TYPE_ALIASES)
if "max_weight_l2" not in beads.columns:
    raise ValueError("beads missing max_weight_l2; rerun Module 5a to regenerate parquet")
n_before = len(beads)
beads = beads[beads["max_weight_l2"] >= FIG2C_MIN_WEIGHT_L2].copy()
print(f"RCTD l2 weight filter (>={FIG2C_MIN_WEIGHT_L2}%): {n_before:,} -> {len(beads):,} beads")
print(beads.groupby(["donor", "puck_region"]).size().unstack(fill_value=0))


def assign_axis_regions(df: pd.DataFrame) -> pd.DataFrame:
    """Cx = cortex; medulla pucks ordered by inner−outer medulla composition depth score,
    then bead-count tertiles → OM / O-IM / IM (whole-puck assignment; one puck ≈ one depth).
    Per-puck coordinates are independent; cross-puck SVD is invalid, hence composition ordering."""
    parts = []
    for donor, g in df.groupby("donor"):
        g = g.copy()
        g["axis_region"] = pd.NA
        g.loc[g["puck_region"] == "Kidney Cortex", "axis_region"] = "Cx"
        g.loc[g["puck_region"] == "Kidney Cortex / Medulla", "axis_region"] = "OM"
        med = g["puck_region"] == "Kidney Medulla"
        if med.any():
            sub = g.loc[med]
            # Per-puck depth score = inner-medulla fraction − outer-medulla fraction
            score = {}
            counts = {}
            for puck, ps in sub.groupby("puck"):
                vc = ps["cell_type"].value_counts(normalize=True)
                score[puck] = float(vc.reindex(FIG2C_INNER_MED).sum()
                                    - vc.reindex(FIG2C_OUTER_MED).sum())
                counts[puck] = len(ps)
            # Sort by depth (outer→inner medulla); bead-weighted tertile split
            order = sorted(score, key=lambda p: score[p])
            total = sum(counts.values())
            cum = 0
            region_map = {}
            for p in order:
                mid = (cum + counts[p] / 2) / total
                region_map[p] = "OM" if mid < 1/3 else ("O-IM" if mid < 2/3 else "IM")
                cum += counts[p]
            g.loc[med, "axis_region"] = sub["puck"].map(region_map).values
            # Print per-donor puck→region mapping for inspection
            dbg = pd.DataFrame({"puck": list(score), "depth": list(score.values()),
                                "n": [counts[p] for p in score],
                                "region": [region_map[p] for p in score]})
            print(f"  [donor {donor}] medullary puck assignment:")
            print(dbg.sort_values("depth").to_string(index=False))
        parts.append(g)
    out = pd.concat(parts, ignore_index=True)
    return out[out["axis_region"].notna()].copy()


beads = assign_axis_regions(beads)
print("\nBeads per donor × axis region:")
print(beads.groupby(["donor", "axis_region"]).size().unstack(fill_value=0))

heat_rows = []
for (donor, region), sub in beads.groupby(["donor", "axis_region"], observed=True):
    counts = sub["cell_type"].value_counts()
    freq = (counts / counts.sum()).reset_index()
    freq.columns = ["cell_type", "fraction"]
    freq["donor"] = str(donor)
    freq["axis_region"] = str(region)
    heat_rows.append(freq)

heatmap_df = pd.concat(heat_rows, ignore_index=True)
heatmap_df.to_csv(CACHED_FIG2C_HEATMAP, index=False)
print("\nHeatmap matrix:", CACHED_FIG2C_HEATMAP)

heatmap_mean = (
    heatmap_df.groupby(["axis_region", "cell_type"], observed=True)["fraction"]
    .mean()
    .reset_index()
)
print("Top cell types per region (donor mean):")
for region in FIG2C_AXIS_REGIONS:
    top = heatmap_mean[heatmap_mean["axis_region"] == region].nlargest(5, "fraction")
    print(region, top[["cell_type", "fraction"]].to_string(index=False))


### Module 5c — Figure 2c rendering (heatmap + spatial maps + markers)

Outputs four figures: heatmap, cell-type spatial map, marker-gene triptych, and composite panel.


In [ ]:
# ── Fig. 2c constants fallback ──
if "FIG2C_MIN_WEIGHT_L2" not in globals():
    FIG2C_DONORS = ["201229", "210412", "210113"]
    FIG2C_AXIS_REGIONS = ["Cx", "OM", "O-IM", "IM"]
    FIG2C_AXIS_LABELS = ["Cx", "OM", "O/IM", "IM"]
    FIG2C_CELL_TYPE_ALIASES = {"MYOF": "MyoF"}
    FIG2C_MIN_WEIGHT_L2 = 50.0
    FIG2C_PAPER_CELL_ORDER = [
        "PEC", "POD", "MC", "EC-GC", "REN", "EC-AEA", "VSMC",
        "PT-S1", "PT-S2", "PT-S3", "aPT", "EC-PTC", "EC-LYM", "FIB", "MyoF", "aFIB",
        "C-TAL", "aTAL1", "MD", "DCT", "CNT", "C-PC", "C-IC-A", "IC-B",
        "DTL1", "DTL2", "EC-DVR", "VSMC/P", "M-TAL", "M-PC", "M-IC-A",
        "EC-AVR", "DTL3", "ATL", "IMCD", "M-FIB",
    ]
    CACHED_FIG2C_HEATMAP = PROCESSED_DATA_DIR / "fig2c_heatmap_matrix.csv"
    FIG2C_HEATMAP_OUTPUT = FIGURES_DIR / "figure2c_heatmap.png"
    CACHED_FIG2C_SPATIAL = PROCESSED_DATA_DIR / "fig2c_spatial_crop.parquet"
    FIG2C_SPATIAL_TYPES_OUTPUT = FIGURES_DIR / "figure2c_spatial_celltypes.png"
    FIG2C_MARKERS = ["SLC12A1", "SLC14A1", "SH3GL3"]
    FIG2C_MARKERS_OUTPUT = FIGURES_DIR / "figure2c_spatial_markers.png"
    FIG2C_COMBINED_OUTPUT = FIGURES_DIR / "figure2c_combined.png"
else:
    FIG2C_AXIS_LABELS = ["Cx", "OM", "O/IM", "IM"]

FIG2C_HEATMAP_GAMMA = 1.8     # >1 increases contrast
FIG2C_HEATMAP_NORM = "max"    # divide by row maximum
FIG2C_PRESENCE_FLOOR = 0.0    # 0 = no confidence attenuation (retain rare cortical types in Cx)

%matplotlib inline
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def _step(msg):
    print(f"[5c {time.strftime('%H:%M:%S')}] {msg}", flush=True)

_step(f"=== 5c heatmap ({FIG2C_HEATMAP_NORM}+gamma={FIG2C_HEATMAP_GAMMA}, floor={FIG2C_PRESENCE_FLOOR}) ===")
if not CACHED_FIG2C_HEATMAP.exists():
    raise FileNotFoundError(f"Missing {CACHED_FIG2C_HEATMAP}")

heatmap_df = pd.read_csv(CACHED_FIG2C_HEATMAP)
heatmap_df["donor"] = heatmap_df["donor"].astype(str)
heatmap_mean = (
    heatmap_df.groupby(["axis_region", "cell_type"], observed=True)["fraction"]
    .mean().reset_index()
)
cx = heatmap_mean[heatmap_mean["axis_region"] == "Cx"].set_index("cell_type")["fraction"]
pt_s1, cnt_cx = float(cx.get("PT-S1", 0)), float(cx.get("CNT", 0))
_step(f"Data check Cx: PT-S1={pt_s1:.3f}, CNT={cnt_cx:.3f}")
if cnt_cx > 0.15 and pt_s1 < 0.15:
    raise RuntimeError("Heatmap CSV is stale; run Module 5b first")

cell_order = [c for c in FIG2C_PAPER_CELL_ORDER if c in heatmap_mean["cell_type"].values]
mat = heatmap_mean.pivot_table(
    index="cell_type", columns="axis_region", values="fraction", fill_value=0
).reindex(index=cell_order, columns=FIG2C_AXIS_REGIONS, fill_value=0)

row_max = mat.max(axis=1)
mat_norm = mat.div(row_max.where(row_max > 0, 1.0), axis=0)
mat_disp = mat_norm ** FIG2C_HEATMAP_GAMMA
if FIG2C_PRESENCE_FLOOR > 0:
    conf = (row_max / FIG2C_PRESENCE_FLOOR).clip(upper=1.0)
    mat_disp = mat_disp.mul(conf, axis=0)
    faint = sorted(row_max[row_max < FIG2C_PRESENCE_FLOOR].index.tolist())
    print(f"[Detection floor] max frequency <{FIG2C_PRESENCE_FLOOR} dimmed: {faint}")
_step(f"Matrix {mat_disp.shape[0]}×{mat_disp.shape[1]}")

# Halve width (5.0→2.6); linewidths=0 removes white grid lines
fig, ax = plt.subplots(figsize=(2.6, 9))
sns.heatmap(
    mat_disp, ax=ax, cmap="viridis", vmin=0, vmax=1,
    linewidths=0, linecolor="none", rasterized=True,
    cbar_kws={"label": "Mean cell type frequency", "shrink": 0.55},
    xticklabels=FIG2C_AXIS_LABELS, yticklabels=True,
)
ax.set_title("Mean cell type frequency", fontsize=11)
ax.set_xlabel(""); ax.set_ylabel("")
ax.tick_params(axis="x", rotation=0, labelsize=9)
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
FIG2C_HEATMAP_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG2C_HEATMAP_OUTPUT, dpi=300, bbox_inches="tight", pad_inches=0.12)
_step(f"Saved {FIG2C_HEATMAP_OUTPUT.name}")
plt.show()
plt.close(fig)
_step("Heatmap complete ✓")


In [ ]:
# ── 5c spatial panels: subclass + RNA + weight + gene maps (aligned with manuscript Fig. 2c right) ──
if "FIG2C_MIN_WEIGHT_L2" not in globals():
    raise RuntimeError("Run path cell / 5c heatmap cell first")

%matplotlib inline
import time
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd

def _step(msg):
    print(f"[5c {time.strftime('%H:%M:%S')}] {msg}", flush=True)

_step("=== 5c spatial: load ===")
sp = pd.read_parquet(CACHED_FIG2C_SPATIAL)
sp["cell_type"] = sp["cell_type"].replace(FIG2C_CELL_TYPE_ALIASES)
_step(f"{len(sp):,} beads, columns: {sp.columns.tolist()}")

# Rotate so M-TAL→ATL depth axis is vertical; crop axis-aligned rectangle in rotated frame
xy = sp[["x", "y"]].to_numpy(float)
mtal_c = xy[sp["cell_type"].values == "M-TAL"].mean(axis=0)
atl_c = xy[sp["cell_type"].values == "ATL"].mean(axis=0)
vec = mtal_c - atl_c
ang = np.pi / 2 - np.arctan2(vec[1], vec[0])  # align vec to +y (M-TAL above, ATL below)
c, s = np.cos(ang), np.sin(ang)
R = np.array([[c, -s], [s, c]])
center = (mtal_c + atl_c) / 2.0
dist = float(np.linalg.norm(vec))           # medullary axis length for crop scale
xyr = (xy - center) @ R.T
sp["px"], sp["py"] = xyr[:, 0], xyr[:, 1]
W = dist * 0.50   # half-width (narrow)
H = dist * 1.25   # half-height (tall) → aspect ~1:2.5
strip = sp[(sp["px"].abs() <= W) & (sp["py"].abs() <= H)].copy()
_step(f"Rotate {np.degrees(ang):.0f}° + rectangular crop (W={W:.0f},H={H:.0f}): {len(sp):,} -> {len(strip):,} beads")

# Cell-type panel uses weight-filtered beads for denoising
strip_hi = strip[strip["max_weight_l2"] >= FIG2C_MIN_WEIGHT_L2] if "max_weight_l2" in strip else strip

CMAP_R = LinearSegmentedColormap.from_list("kr", ["black", "#FF2A2A"])
CMAP_B = LinearSegmentedColormap.from_list("kb", ["black", "#3B7BFF"])
CMAP_G = LinearSegmentedColormap.from_list("kg", ["black", "#33E03A"])

TYPE_COLORS = {
    "M-TAL": "#FF3B30", "ATL": "#33E03A", "EC-DVR": "#3B6BFF", "EC-AVR": "#79A6FF",
    "DTL1": "#FF9F0A", "DTL2": "#FFC04D", "DTL3": "#FFD98A",
    "IMCD": "#9BE06A", "M-PC": "#C77DFF", "M-IC-A": "#FF7AD5", "M-FIB": "#00C2C7",
    "C-TAL": "#B03030", "VSMC/P": "#888888",
}
DEFAULT_C = "#555555"

def _panel(ax, title=""):
    ax.set_facecolor("black"); ax.set_xticks([]); ax.set_yticks([])
    for sp_ in ax.spines.values(): sp_.set_visible(False)
    ax.set_aspect("equal")
    if title: ax.set_title(title, fontsize=9)

def plot_subclass(ax, df):
    _panel(ax, "Subclass")
    hi = ["EC-AVR","DTL1","DTL2","DTL3","IMCD","EC-DVR","ATL","M-TAL"]
    for ct in df["cell_type"].value_counts().index:
        if ct in hi: continue
        m = df["cell_type"] == ct
        ax.scatter(df.loc[m,"px"], df.loc[m,"py"], s=1.3, c=TYPE_COLORS.get(ct, DEFAULT_C), linewidths=0, rasterized=True)
    for ct in hi:
        m = df["cell_type"] == ct
        if m.any(): ax.scatter(df.loc[m,"px"], df.loc[m,"py"], s=1.6, c=TYPE_COLORS.get(ct, DEFAULT_C), linewidths=0, rasterized=True)

def plot_rna(ax, df):
    _panel(ax, "RNA")
    def nrm(gene, vmax):
        v = df[gene].to_numpy(float) if gene in df.columns else np.zeros(len(df))
        return np.clip(v / vmax, 0, 1) ** 0.6
    cols = np.stack([nrm("SLC12A1",25), nrm("SH3GL3",3), nrm("SLC14A1",4)], axis=1)
    o = np.argsort(cols.max(axis=1))
    ax.scatter(df["px"].to_numpy()[o], df["py"].to_numpy()[o], s=1.3, c=cols[o], linewidths=0, rasterized=True)
    for i,(g,col) in enumerate([("SLC12A1","#FF3B30"),("SH3GL3","#33E03A"),("SLC14A1","#5B8BFF")]):
        ax.text(0.04, 0.99-0.05*i, g, color=col, fontsize=7, style="italic", ha="left", va="top", transform=ax.transAxes)

def plot_field(ax, df, col, cmap, vmax, title, ticks):
    _panel(ax, title)
    v = df[col].to_numpy(float) if col in df.columns else np.zeros(len(df))
    o = np.argsort(v)
    sc = ax.scatter(df["px"].to_numpy()[o], df["py"].to_numpy()[o], c=v[o], cmap=cmap,
                    vmin=0, vmax=vmax, s=1.6, linewidths=0, rasterized=True)
    cb = plt.colorbar(sc, ax=ax, fraction=0.05, pad=0.04, ticks=ticks)
    cb.ax.tick_params(labelsize=7)

_step("Rendering multi-panel figure")
fig = plt.figure(figsize=(13, 6.5), facecolor="white")
gs = gridspec.GridSpec(2, 5, width_ratios=[1.4,1.4,1,1,1], height_ratios=[1,1], wspace=0.25, hspace=0.2)
plot_subclass(fig.add_subplot(gs[:,0]), strip_hi)
plot_rna(fig.add_subplot(gs[:,1]), strip)
plot_field(fig.add_subplot(gs[0,2]), strip, "w_M-TAL", CMAP_R, 80, "M-TAL", [0,40,80])
plot_field(fig.add_subplot(gs[0,3]), strip, "w_EC-DVR", CMAP_B, 80, "EC-DVR", [0,40,80])
plot_field(fig.add_subplot(gs[0,4]), strip, "w_ATL", CMAP_G, 80, "ATL", [0,40,80])
plot_field(fig.add_subplot(gs[1,2]), strip, "SLC12A1", CMAP_R, 25, "SLC12A1", [0,25])
plot_field(fig.add_subplot(gs[1,3]), strip, "SLC14A1", CMAP_B, 4, "SLC14A1", [0,4])
plot_field(fig.add_subplot(gs[1,4]), strip, "SH3GL3", CMAP_G, 3, "SH3GL3", [0,3])
FIG2C_SPATIAL_TYPES_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG2C_SPATIAL_TYPES_OUTPUT, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
_step(f"Saved {FIG2C_SPATIAL_TYPES_OUTPUT.name}")
plt.show()
plt.close(fig)
_step("Spatial panels complete ✓")


---
## Module 6 — Figure 2e reproduction

Manuscript Fig. 2e: **magnified renal corpuscle** on a Slide-seq2 cortical puck, split into upper and lower panels:
1. **Upper**: predicted corpuscle-associated cell types — POD, EC-GC, MD, PEC, REN (white background)
2. **Lower**: scaled marker-gene RGB — *EMCN* (blue, 0–6), *NOS1* (green, 0–10), *REN* (red, 0–6) (black background)

> **Representative puck**: `Puck_200903_06` (GSM5554452, cortex) — identified from the **Extended Data Fig. 4c** panel labeled "2e".
>
> Scale bar: 100 µm. Data reuse Slide-seq Giotto RDS from Module 5; no additional download required.
>
> **Workflow** (data validation before plotting): 6a → 6b → 6c.


### Module 6a — Data validation (confirm inputs before plotting)

Pre-flight checks:
1. **Representative puck `Puck_200903_06` is present** (manifest + RDS file);
2. **Five corpuscle cell types (POD / EC-GC / MD / PEC / REN) are resolved by RCTD** (high-confidence bead counts);
3. **Marker genes `EMCN` / `NOS1` / `REN` are present** in the expression matrix (raw count scale).

Proceed to 6b only if all three checks pass.


In [ ]:
# Module 6a — Figure 2e path constants + data validation
import gzip, io, warnings
from pathlib import Path
import numpy as np, pandas as pd
!pip install -q rdata pyarrow
import rdata, scipy.sparse as sparse
warnings.filterwarnings("ignore", category=UserWarning, module="rdata")

# Reuse Module 5 Slide-seq paths (auto-defined if Module 5 path cell was skipped)
if "SLIDESEQ_PUCKS" not in globals():
    RAW_DATA_DIR = DATA_DIR / "raw"
    SLIDESEQ_DIR = RAW_DATA_DIR / "slideseq" / "GSE183274"
    SLIDESEQ_PUCKS = SLIDESEQ_DIR / "pucks"
    SLIDESEQ_MANIFEST = SLIDESEQ_DIR / "metadata" / "sample_manifest.tsv"

# ── Figure 2e constants ──
FIG2E_PUCK = "Puck_200903_06"          # from Extended Data Fig. 4c panel "2e"
FIG2E_RC_TYPES = ["POD", "EC-GC", "MD", "PEC", "REN"]   # corpuscle-associated types
FIG2E_MARKERS = ["EMCN", "NOS1", "REN"]                  # lower-panel markers
FIG2E_MARKER_VMAX = {"EMCN": 6, "NOS1": 10, "REN": 6}    # manuscript color-scale maxima
FIG2E_MIN_WEIGHT_L2 = 50.0
CACHED_FIG2E_SPATIAL = PROCESSED_DATA_DIR / "fig2e_spatial.parquet"
FIG2E_TYPES_OUTPUT = FIGURES_DIR / "figure2e_celltypes.png"
FIG2E_RNA_OUTPUT = FIGURES_DIR / "figure2e_rna.png"
FIG2E_COMBINED_OUTPUT = FIGURES_DIR / "figure2e_combined.png"

manifest = pd.read_csv(SLIDESEQ_MANIFEST, sep="\t")

print("="*56)
print("Figure 2e data validation")
print("="*56)

# ① Representative puck present
row = manifest[manifest["puck"] == FIG2E_PUCK]
assert len(row) == 1, f"{FIG2E_PUCK} not found in manifest"
rds = SLIDESEQ_PUCKS / row["rds_file"].iloc[0]
print(f"① Representative puck: {FIG2E_PUCK}  ({row['gsm'].iloc[0]}, {row['region'].iloc[0]})")
print(f"   RDS file: {rds.name}  exists={rds.exists()}")
assert rds.exists(), "RDS file missing; download required"

# Load Giotto object
raw = gzip.decompress(gzip.decompress(rds.read_bytes()))
g = rdata.conversion.convert(rdata.parser.parse_file(io.BytesIO(raw)))
meta = pd.DataFrame(g.cell_metadata); meta.columns = [str(c) for c in meta.columns]
locs = pd.DataFrame(g.spatial_locs); locs.columns = [str(c) for c in locs.columns]

# ② Five corpuscle cell types resolved by deconvolution
ct = meta["maxCelltype.l2"].astype(str).str.replace(r"^l[12]\.", "", regex=True)
w = pd.to_numeric(meta["maxWeight.l2"], errors="coerce")
hi_ct = ct[w >= FIG2E_MIN_WEIGHT_L2]
print(f"\n② Full puck beads: {len(meta):,}, high-confidence (>={FIG2E_MIN_WEIGHT_L2:.0f}%): {len(hi_ct):,}")
rc_counts = hi_ct[hi_ct.isin(FIG2E_RC_TYPES)].value_counts().reindex(FIG2E_RC_TYPES, fill_value=0)
print("   Corpuscle-type high-confidence bead counts:")
for t in FIG2E_RC_TYPES:
    print(f"     {t:6s}: {int(rc_counts[t])}")
assert (rc_counts[["POD", "EC-GC"]] > 20).all(), "Too few POD/EC-GC beads to localize corpuscles"

# ③ Marker genes present in expression matrix
trip = g.raw_exprs
genes_upper = [str(x).upper() for x in trip.Dimnames[0]]
print("\n③ Marker gene check (raw counts):")
missing = []
for gene in FIG2E_MARKERS:
    ok = gene.upper() in genes_upper
    print(f"     {gene}: {'✓ present' if ok else '✗ missing'}")
    if not ok: missing.append(gene)
assert not missing, f"Missing genes: {missing}"

print("\n✅ Validation passed: no additional download; proceed to Module 6b corpuscle localization")


### Module 6b — Renal corpuscle cluster localization

The manuscript "2e" crop lies in the upper-left of `Puck_200903_06`, spanning **two adjacent corpuscles plus MD**. Steps:
1. Extract high-confidence beads and coordinates for the full puck; cache as `fig2e_spatial.parquet`;
2. Apply `DBSCAN` on **POD** beads → each cluster approximates one corpuscle;
3. Plot corpuscle distribution with cluster IDs for visual selection of the 6c crop window.


In [ ]:
# Module 6b — extract Puck_200903_06 spatial table + DBSCAN corpuscle clusters
import gzip, io, rdata, pandas as pd, numpy as np, scipy.sparse as sparse
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# ── Extract full-puck spatial table (coordinates, types, weights, marker raw counts) ──
def build_fig2e_spatial(rds_path, markers):
    raw = gzip.decompress(gzip.decompress(rds_path.read_bytes()))
    g = rdata.conversion.convert(rdata.parser.parse_file(io.BytesIO(raw)))
    meta = pd.DataFrame(g.cell_metadata); meta.columns = [str(c) for c in meta.columns]
    locs = pd.DataFrame(g.spatial_locs); locs.columns = [str(c) for c in locs.columns]
    id_col = "cell_ID" if "cell_ID" in meta.columns else meta.columns[0]
    df = meta.merge(locs, on=id_col, how="inner") if id_col in locs.columns else pd.concat(
        [meta.reset_index(drop=True), locs.reset_index(drop=True)], axis=1)
    for xc, yc in [("sdimx", "sdimy"), ("x", "y"), ("imagecol", "imagerow")]:
        if xc in df.columns and yc in df.columns:
            break
    n = len(df)
    out = pd.DataFrame({
        "x": pd.to_numeric(df[xc], errors="coerce"),
        "y": pd.to_numeric(df[yc], errors="coerce"),
        "cell_type": df["maxCelltype.l2"].astype(str).str.replace(r"^l[12]\.", "", regex=True),
        "max_weight_l2": pd.to_numeric(df["maxWeight.l2"], errors="coerce"),
    })
    trip = g.raw_exprs
    gu = [str(s).upper() for s in trip.Dimnames[0]]
    mat = sparse.coo_matrix((trip.x, (trip.i, trip.j)), shape=tuple(trip.Dim)).tocsr()
    for gene in markers:
        out[gene] = np.asarray(mat.getrow(gu.index(gene.upper())).toarray()).ravel()[:n]
    return out.dropna(subset=["x", "y"])

rds = SLIDESEQ_PUCKS / manifest.loc[manifest["puck"] == FIG2E_PUCK, "rds_file"].iloc[0]
sp = build_fig2e_spatial(rds, FIG2E_MARKERS)
sp.to_parquet(CACHED_FIG2E_SPATIAL, index=False)
print(f"spatial table: {sp.shape} -> {CACHED_FIG2E_SPATIAL.name}")

hi = sp[sp["max_weight_l2"] >= FIG2E_MIN_WEIGHT_L2].copy()
RC_C = {"POD": "#E8A0BF", "EC-GC": "#3B6BFF", "MD": "#33C03A", "PEC": "#8B5A2B", "REN": "#7B2FBE"}

# ── POD DBSCAN → corpuscle clusters (one cluster ≈ one corpuscle) ──
pod = hi.loc[hi["cell_type"] == "POD", ["x", "y"]].to_numpy()
lab = DBSCAN(eps=150, min_samples=8).fit(pod).labels_
clusters = []
for l in sorted(set(lab) - {-1}):
    pts = pod[lab == l]
    clusters.append({"cluster": l, "n_pod": len(pts), "cx": pts[:, 0].mean(), "cy": pts[:, 1].mean()})
clusters = pd.DataFrame(clusters).sort_values("n_pod", ascending=False).reset_index(drop=True)
print(f"\nPOD clusters: {len(clusters)} (sorted by POD bead count)")
print(clusters.head(10).to_string(index=False))

# ── Diagnostic plot (English titles to avoid missing CJK fonts) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
ax = axes[0]; ax.set_title(f"{FIG2E_PUCK} — high-conf beads (grey) + renal corpuscle types", fontsize=10)
ax.scatter(hi["x"], hi["y"], s=2, c="#DDDDDD", linewidths=0)
for t in FIG2E_RC_TYPES:
    m = hi["cell_type"] == t
    ax.scatter(hi.loc[m, "x"], hi.loc[m, "y"], s=6, c=RC_C[t], label=t, linewidths=0)
ax.legend(markerscale=2, fontsize=8, loc="upper right"); ax.set_aspect("equal"); ax.invert_yaxis()
ax2 = axes[1]; ax2.set_title("POD DBSCAN clusters (labelled)", fontsize=10)
ax2.scatter(hi["x"], hi["y"], s=2, c="#EEEEEE", linewidths=0)
for _, r in clusters.iterrows():
    pts = pod[lab == r["cluster"]]
    ax2.scatter(pts[:, 0], pts[:, 1], s=8, linewidths=0)
    ax2.text(r["cx"], r["cy"], str(int(r["cluster"])), fontsize=9, fontweight="bold")
ax2.set_aspect("equal"); ax2.invert_yaxis()
plt.tight_layout(); plt.show()


### Module 6c — Figure 2e rendering

After selecting a corpuscle cluster, crop a local window (~100 µm) and render two panels:
- **Upper**: cell types (POD / EC-GC / MD / PEC / REN; white background)
- **Lower**: *EMCN* / *NOS1* / *REN* RGB composite (black background; scales 0–6 / 0–10 / 0–6)


In [ ]:
# Module 6c — render Figure 2e (clusters 5+10 corpuscle crop + color bars)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colorbar as mcolorbar
from matplotlib.colors import LinearSegmentedColormap, Normalize
import numpy as np, pandas as pd
from sklearn.cluster import DBSCAN
from scipy.spatial import cKDTree

FIG2E_CLUSTERS = (5, 10)
FIG2E_WIN_SCALE = 0.8
FIG2E_WIN_MIN_HALF = 360
FIG2E_FAINT_MIN = 20
# Display vmax (NOS1 is very sparse on this puck, max=2; lower vmax to brighten green)
FIG2E_RNA_VMAX_DISP = {"EMCN": 6, "NOS1": 2.5, "REN": 6}
# Color-bar ranges (aligned with manuscript)
FIG2E_CBAR_RANGE = {"EMCN": 6, "NOS1": 10, "REN": 6}

sp = pd.read_parquet(CACHED_FIG2E_SPATIAL)
sp["cell_type"] = sp["cell_type"].replace({"MYOF": "MyoF"})
hi = sp[sp["max_weight_l2"] >= FIG2E_MIN_WEIGHT_L2].copy()

# ── Select two clusters → crop window ──
pod = hi.loc[hi["cell_type"] == "POD", ["x", "y"]].to_numpy()
lab = DBSCAN(eps=150, min_samples=8).fit(pod).labels_
cen = {l: pod[lab == l].mean(axis=0) for l in set(lab) - {-1}}
c_a, c_b = cen[FIG2E_CLUSTERS[0]], cen[FIG2E_CLUSTERS[1]]
mid = (c_a + c_b) / 2
half = max(np.linalg.norm(c_a - c_b) * FIG2E_WIN_SCALE, FIG2E_WIN_MIN_HALF)
x0, x1, y0, y1 = mid[0]-half, mid[0]+half, mid[1]-half, mid[1]+half
crop = sp[(sp.x >= x0) & (sp.x <= x1) & (sp.y >= y0) & (sp.y <= y1)].copy()
crop_hi = crop[crop.max_weight_l2 >= FIG2E_MIN_WEIGHT_L2]
print(f"Window side {2*half:.0f} units, crop {len(crop)} beads")
print(f"NOS1 (green) very sparse on this puck: max={crop['NOS1'].max():.0f}, >0 only {int((crop['NOS1']>0).sum())} beads → display vmax set to {FIG2E_RNA_VMAX_DISP['NOS1']}")

# ── Scale bar ──
xy_all = sp[["x", "y"]].to_numpy()
d, _ = cKDTree(xy_all).query(xy_all[:2000], k=2)
pitch = np.median(d[:, 1]); um_per_unit = 10.0 / pitch; bar_units = 100.0 / um_per_unit

# ── Color mapping ──
RC_C = {"POD": "#E8669E", "EC-GC": "#2E6BFF", "MD": "#22C55E", "PEC": "#7A4A2B", "REN": "#9333EA"}
RC_FAINT = {"POD": "#F6C9DC", "EC-GC": "#B7CBFF", "MD": "#B7E8C6", "PEC": "#CBB6A6", "REN": "#D9BCF0"}
CMAP_B = LinearSegmentedColormap.from_list("kb", ["black", "#3B7BFF"])
CMAP_G = LinearSegmentedColormap.from_list("kg", ["black", "#33E03A"])
CMAP_R = LinearSegmentedColormap.from_list("kr", ["black", "#FF3030"])

def _scalebar(ax, dark):
    xb0 = x0 + (x1 - x0) * 0.06; yb = y1 - (y1 - y0) * 0.08
    ax.plot([xb0, xb0 + bar_units], [yb, yb], "-", color="white" if dark else "black", lw=2.5)

# ── Upper panel: cell types ──
def plot_types(ax):
    ax.set_facecolor("white")
    order = ["PEC", "REN", "MD", "EC-GC", "POD"]
    faint = crop[(crop.max_weight_l2 >= FIG2E_FAINT_MIN) & (crop.max_weight_l2 < FIG2E_MIN_WEIGHT_L2)]
    for t in order:
        m = faint.cell_type == t
        if m.any(): ax.scatter(faint.loc[m, "x"], faint.loc[m, "y"], s=16, c=RC_FAINT[t], linewidths=0, rasterized=True)
    for t in order:
        m = crop_hi.cell_type == t
        if m.any(): ax.scatter(crop_hi.loc[m, "x"], crop_hi.loc[m, "y"], s=26, c=RC_C[t], label=t, linewidths=0, rasterized=True)
    ax.legend(markerscale=1.3, fontsize=8, loc="lower right", framealpha=0.9)
    _scalebar(ax, dark=False)
    ax.set_title("Predicted cell types (renal corpuscle)", fontsize=10)

# ── Middle panel: RGB ──
def plot_rna(ax):
    ax.set_facecolor("black")
    def nrm(gene):
        v = crop[gene].to_numpy(float)
        return np.clip(v / FIG2E_RNA_VMAX_DISP[gene], 0, 1) ** 0.6
    cols = np.stack([nrm("REN"), nrm("NOS1"), nrm("EMCN")], axis=1)  # R=REN,G=NOS1,B=EMCN
    o = np.argsort(cols.max(axis=1))
    ax.scatter(crop.x.to_numpy()[o], crop.y.to_numpy()[o], s=15, c=cols[o], linewidths=0, rasterized=True)
    _scalebar(ax, dark=True)
    ax.set_title("Marker gene expression (scaled)", fontsize=10)

def _fmt(ax):
    ax.set_aspect("equal"); ax.set_xlim(x0, x1); ax.set_ylim(y1, y0)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_color("black")

fig = plt.figure(figsize=(5.5, 11), facecolor="white")
gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1, 0.10], hspace=0.14)
plot_types(fig.add_subplot(gs[0])); _fmt(fig.gca())
plot_rna(fig.add_subplot(gs[1])); _fmt(fig.gca())

# ── Lower panel: three gradient color bars ──
cbar_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[2], wspace=0.6)
for i, (gene, cmap) in enumerate([("EMCN", CMAP_B), ("NOS1", CMAP_G), ("REN", CMAP_R)]):
    cax = fig.add_subplot(cbar_gs[i])
    vmax = FIG2E_CBAR_RANGE[gene]
    cb = mcolorbar.ColorbarBase(cax, cmap=cmap, norm=Normalize(0, vmax), orientation="horizontal", ticks=[0, vmax])
    cb.set_label(gene, fontsize=9, style="italic")
    cb.ax.tick_params(labelsize=8)

FIG2E_COMBINED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG2E_COMBINED_OUTPUT, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print("Saved", FIG2E_COMBINED_OUTPUT.name)
plt.show(); plt.close(fig)
print("Figure 2e complete ✓")


---
## Module 7 — Figure 2f reproduction

Manuscript Fig. 2f: **marker-gene dot plot** across three platforms, assessing cell-type marker consistency in spatial and single-nucleus assays.

- **Three panels**: snCv3/scCv3 (blue), Slide-seq2 (orange), Visium (purple)
- **Rows**: nine marker genes — `NPHS2 EMCN POSTN VCAM1 SLC5A12 PALMD MYH11 REN NOS1`
- **Columns**: nine cell types — `POD EC-GC MC PEC PT-S1/2 EC-AEA VSMC REN MD`
- **Dot size**: fraction of cells/spots expressing the gene (Exp. %)
- **Dot color**: mean expression z-score within each cell type (gene-wise across types, clipped to 0–2)

**Data sources**
- snCv3: CELLxGENE integrated `kidney_integrated_sc_snRNA.h5ad` (includes `subclass.l2` annotations)
- Slide-seq2: GSE183274 Giotto pucks (RCTD deconvolution; `maxWeight.l2 ≥ 50`)
- Visium: GSE183456 Space Ranger output (raw matrices from GEO; **label transfer** required for spot annotations)


### Module 7a — Data validation + Visium Space Ranger download

Confirm three-platform inputs before plotting:
1. **snCv3**: verify `kidney_integrated_sc_snRNA.h5ad` exists and contains all nine marker genes and target `subclass.l2` types.
2. **Slide-seq2**: verify Giotto pucks and manifest are available.
3. **Visium**: if missing locally, download 23 sample Space Ranger archives from GEO super-series GSE183279 sub-series **GSE183456** (10x H5 matrices, `*.tar.gz`). Downloads are resumable; existing files are skipped.


In [ ]:
# Module 7a — data validation + Visium Space Ranger download
import re, urllib.request
from pathlib import Path
import anndata as ad

FIG2F_GENES = ['NPHS2','EMCN','POSTN','VCAM1','SLC5A12','PALMD','MYH11','REN','NOS1']
FIG2F_TYPES = ['POD','EC-GC','MC','PEC','PT-S1/2','EC-AEA','VSMC','REN','MD']
SNC_H5AD   = RAW_DATA_DIR / 'cellxgene' / 'kidney_integrated_sc_snRNA.h5ad'
VISIUM_DIR = RAW_DATA_DIR / 'visium' / 'GSE183456'
VISIUM_DIR.mkdir(parents=True, exist_ok=True)

# 1) snCv3 validation
A = ad.read_h5ad(SNC_H5AD, backed='r')
sym = set(A.var['feature_name'].astype(str))
sub = set(A.obs['subclass.l2'].astype(str).unique())
print('snCv3:', A.shape,
      '| missing genes:', [g for g in FIG2F_GENES if g not in sym],
      '| missing types:', [t for t in FIG2F_TYPES if t not in sub and t!='PT-S1/2'])
del A

# 2) Slide-seq2 validation
print('Slide-seq2 pucks:', len(list(SLIDESEQ_PUCKS.glob('*.rds.gz'))), '| manifest:', SLIDESEQ_MANIFEST.exists())

# 3) Visium download (GSE183456, 23 samples GSM6047774–GSM6047796)
GSM_IDS = [f'GSM{n}' for n in range(6047774, 6047797)]
def suppl_listing(gsm):
    folder = gsm[:-3] + 'nnn'
    url = f'https://ftp.ncbi.nlm.nih.gov/geo/samples/{folder}/{gsm}/suppl/'
    html = urllib.request.urlopen(url, timeout=60).read().decode('utf-8', 'ignore')
    names = [f.split('/')[-1] for f in re.findall(r'href="([^"]+\.tar\.gz)"', html)]
    return url, names

done = {p.name for p in VISIUM_DIR.glob('*.tar.gz')}
print(f'Downloaded {len(done)}/23; checking missing...')
for gsm in GSM_IDS:
    if any(n.startswith(gsm) for n in done):
        continue
    url, names = suppl_listing(gsm)
    for nm in names:
        dst = VISIUM_DIR / nm
        if dst.exists():
            continue
        print('Downloading', nm)
        urllib.request.urlretrieve(url + nm, dst)
print('Visium tar.gz:', len(list(VISIUM_DIR.glob('*.tar.gz'))), 'files')
print('Module 7a complete')


### Module 7b — Slide-seq2 extraction of nine marker genes

Iterate Giotto pucks and extract per-bead **raw counts** for nine marker genes, `nUMI`, and RCTD maximum-weight cell type (`maxWeight.l2 ≥ 50`; retain only the nine target types; PT-S1 and PT-S2 are merged as PT-S1/2 downstream).

Giotto `raw_exprs` may be `dgTMatrix` (`i,j,x`) or `dgCMatrix` (`i,p,x`); `_to_csr` handles both. Results cached in `fig2f_slideseq_beads.parquet`; skipped if cache exists.


In [ ]:
# Module 7b — Slide-seq2 extraction of nine marker genes (raw counts)
import gzip, io, warnings
import numpy as np, pandas as pd, scipy.sparse as sparse
import rdata
warnings.filterwarnings('ignore', category=UserWarning, module='rdata')

FIG2F_GENES = ['NPHS2','EMCN','POSTN','VCAM1','SLC5A12','PALMD','MYH11','REN','NOS1']
FIG2F_SS_CACHE = PROCESSED_DATA_DIR / 'fig2f_slideseq_beads.parquet'
SS_TARGET = ['POD','EC-GC','MC','PEC','PT-S1','PT-S2','EC-AEA','VSMC','REN','MD']

def read_giotto_puck(path):
    raw = gzip.decompress(gzip.decompress(path.read_bytes()))
    return rdata.conversion.convert(rdata.parser.parse_file(io.BytesIO(raw)))

def _to_csr(node):
    """Support dgTMatrix (i,j,x) and dgCMatrix (i,p,x). Returns genes × beads CSR."""
    dim = tuple(int(d) for d in node.Dim)
    if hasattr(node, 'j'):
        return sparse.coo_matrix((node.x, (node.i, node.j)), shape=dim).tocsr()
    return sparse.csc_matrix((node.x, np.asarray(node.i), np.asarray(node.p)), shape=dim).tocsr()

def _type_col(df):
    for c in ['maxCelltype.l2','subclass.l2','maxCelltype_l2']:
        if c in df.columns: return c
    raise KeyError(list(df.columns)[:20])

def extract_genes_one(path):
    g = read_giotto_puck(path)
    meta = pd.DataFrame(g.cell_metadata); meta.columns = [str(c) for c in meta.columns]
    ct = meta[_type_col(meta)].astype(str).str.replace(r'^l[12]\.','',regex=True)
    w  = pd.to_numeric(meta.get('maxWeight.l2'), errors='coerce').values
    numi = pd.to_numeric(meta.get('nUMI'), errors='coerce').values
    keep = (w>=50) & ct.isin(SS_TARGET).values
    if keep.sum()==0: return None
    node = g.raw_exprs
    gene_names = [str(x).upper() for x in node.Dimnames[0]]
    M = _to_csr(node)
    out = {'cell_type': ct.values[keep], 'nUMI': numi[keep]}
    for gg in FIG2F_GENES:
        if gg in gene_names:
            row = np.asarray(M.getrow(gene_names.index(gg)).todense()).ravel()
            out[gg] = row[keep]
        else:
            out[gg] = np.zeros(int(keep.sum()))
    return pd.DataFrame(out)

if FIG2F_SS_CACHE.exists():
    beads = pd.read_parquet(FIG2F_SS_CACHE)
    print('Cache exists; skipping extraction:', beads.shape)
else:
    manifest = pd.read_csv(SLIDESEQ_MANIFEST, sep='\t')
    parts=[]
    for i,row in manifest.iterrows():
        fp = SLIDESEQ_PUCKS / row['rds_file']
        if not fp.exists(): continue
        df = extract_genes_one(fp)
        if df is not None: parts.append(df)
        if (i+1)%10==0: print(f'  {i+1}/{len(manifest)} pucks')
    beads = pd.concat(parts, ignore_index=True)
    beads.to_parquet(FIG2F_SS_CACHE, index=False)
    print('Saved', FIG2F_SS_CACHE.name, beads.shape)
print('Beads per cell type:'); print(beads['cell_type'].value_counts())
print('Module 7b complete')


### Module 7c — Visium assembly + label transfer

GEO provides Visium raw Space Ranger matrices only; spot-level cell-type labels require transfer:

1. **Assembly**: decompress `filtered_feature_bc_matrix.h5` from 23 `*.tar.gz` archives; concatenate into a single `AnnData`; cache `fig2f_visium_raw.h5ad`.
2. **Transfer**: snCv3 reference (`state == "reference"` healthy cells; ≤800 cells per type) → shared genes → PCA / neighbors / UMAP → `sc.tl.ingest(embedding_method="pca")` to project `subclass.l2` onto Visium spots; cache `fig2f_visium_labeled.parquet`.

> Healthy reference cells only, to avoid mapping spots to degenerative states; PCA embedding (not UMAP) for memory efficiency.


In [ ]:
# Module 7c — Visium assembly + label transfer
import tarfile, tempfile, time, gc
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
from pathlib import Path

VISIUM_DIR    = RAW_DATA_DIR / 'visium' / 'GSE183456'
VISIUM_H5AD   = PROCESSED_DATA_DIR / 'fig2f_visium_raw.h5ad'
VISIUM_LABELS = PROCESSED_DATA_DIR / 'fig2f_visium_labeled.parquet'
SNC_H5AD      = RAW_DATA_DIR / 'cellxgene' / 'kidney_integrated_sc_snRNA.h5ad'
REF_CAP = 800

# 1) Assemble Visium AnnData
if VISIUM_H5AD.exists():
    print('Visium AnnData cache exists; skipping assembly:', f'{VISIUM_H5AD.stat().st_size/1e6:.0f}MB')
else:
    parts=[]
    with tempfile.TemporaryDirectory() as td:
        for tgz in sorted(VISIUM_DIR.glob('*.tar.gz')):
            sample = tgz.name.split('_')[0]
            with tarfile.open(tgz) as tf:
                h5m = [m for m in tf.getmembers() if m.name.endswith('filtered_feature_bc_matrix.h5')]
                if not h5m: continue
                tf.extract(h5m[0], td)
                a = sc.read_10x_h5(Path(td)/h5m[0].name)
            a.var_names_make_unique(); a.obs['sample']=sample
            a.obs_names = [f'{sample}_{b}' for b in a.obs_names]
            parts.append(a); print('  read', sample, a.shape)
    V = ad.concat(parts, join='outer')
    V.write_h5ad(VISIUM_H5AD); print('Saved', VISIUM_H5AD.name, V.shape)

# 2) Label transfer snCv3 (reference state) → Visium
if VISIUM_LABELS.exists():
    lab = pd.read_parquet(VISIUM_LABELS)
    print('Label cache exists; skipping transfer. Top spot types:')
    print(lab['pred_subclass_l2'].value_counts().head(9))
else:
    t0=time.time()
    V = sc.read_h5ad(VISIUM_H5AD)
    sc.pp.normalize_total(V, target_sum=1e4); sc.pp.log1p(V)
    Vgenes = set(map(str, V.var_names))
    A = ad.read_h5ad(SNC_H5AD, backed='r')
    sym = A.var['feature_name'].astype(str).values
    common_cols = np.where(np.isin(sym, list(Vgenes)))[0]
    state = A.obs['state'].astype(str).values; sub = A.obs['subclass.l2'].astype(str).values
    is_ref = state=='reference'; rng = np.random.default_rng(0); idx=[]
    for c in np.unique(sub[is_ref]):
        pos = np.where((sub==c)&is_ref)[0]
        if len(pos)<20: continue
        idx.extend((pos if len(pos)<=REF_CAP else rng.choice(pos,REF_CAP,replace=False)).tolist())
    idx=np.sort(np.array(idx))
    print(f'Reference (healthy) subsample {len(idx)} cells, {len(np.unique(sub[idx]))} types')
    ref = A[idx, common_cols].to_memory()
    ref.var_names = sym[common_cols]; ref.var_names_make_unique()
    del A; gc.collect()
    common=[g for g in ref.var_names if g in Vgenes]
    ref=ref[:,common].copy(); Vc=V[:,common].copy()
    sc.pp.highly_variable_genes(ref, n_top_genes=2000)
    ref=ref[:,ref.var['highly_variable']].copy(); Vc=Vc[:,ref.var_names].copy()
    sc.pp.scale(ref, max_value=10); sc.pp.pca(ref, n_comps=40)
    sc.pp.neighbors(ref, n_neighbors=15); sc.tl.umap(ref)
    ref.obs['subclass.l2']=ref.obs['subclass.l2'].astype('category')
    sc.tl.ingest(Vc, ref, obs='subclass.l2', embedding_method='pca')
    pred=Vc.obs['subclass.l2'].astype(str)
    out=pd.DataFrame({'pred_subclass_l2':pred.values,'sample':V.obs['sample'].values}, index=V.obs_names)
    out.to_parquet(VISIUM_LABELS)
    print(f'ingest complete {time.time()-t0:.0f}s, saved {VISIUM_LABELS.name}')
print('Module 7c complete')


### Module 7d — Three-platform merge and Figure 2f rendering

Compute unified (% expressing, standardized mean expression) per platform: snCv3 uses log-normalized `expm1` means; Slide-seq2 and Visium use CPM-normalized means, then gene-wise z-scoring. Render three-panel dot plot with platform colors (blue / orange / purple); include Exp.(%) size legend and Ave. Exp. color bar; save `figure2f_dotplot.png`.


In [ ]:
# Module 7d — merge three platforms and render Figure 2f
import numpy as np, pandas as pd, scipy.sparse as sps, anndata as ad, scanpy as sc
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import matplotlib.colorbar as mcolorbar
from matplotlib.colors import LinearSegmentedColormap, Normalize

FIG2F_GENES = ['NPHS2','EMCN','POSTN','VCAM1','SLC5A12','PALMD','MYH11','REN','NOS1']
FIG2F_TYPES = ['POD','EC-GC','MC','PEC','PT-S1/2','EC-AEA','VSMC','REN','MD']
FIG2F_OUTPUT = FIGURES_DIR / 'figure2f_dotplot.png'

def _scaled(avg):
    # Gene-wise z-score across cell types (consistent with manuscript scaled expression)
    return (avg.sub(avg.mean(axis=1),axis=0)).div(avg.std(axis=1)+1e-9, axis=0)

def compute_snc():
    h5 = RAW_DATA_DIR / 'cellxgene' / 'kidney_integrated_sc_snRNA.h5ad'
    A = ad.read_h5ad(h5, backed='r')
    sym = A.var['feature_name'].astype(str)
    cols = [int(np.where(sym.values==g)[0][0]) for g in FIG2F_GENES]
    sub = A.obs['subclass.l2'].astype(str).values
    mask = np.isin(sub, FIG2F_TYPES)
    X = A[mask, cols].X; X = X.toarray() if sps.issparse(X) else np.asarray(X)
    lab = sub[mask]
    pct = pd.DataFrame(index=FIG2F_GENES, columns=FIG2F_TYPES, dtype=float)
    avg = pd.DataFrame(index=FIG2F_GENES, columns=FIG2F_TYPES, dtype=float)
    for c in FIG2F_TYPES:
        m = lab==c; pct[c]=(X[m]>0).mean(axis=0)*100; avg[c]=np.expm1(X[m]).mean(axis=0)
    return pct, _scaled(avg)

def compute_ss():
    ss = pd.read_parquet(PROCESSED_DATA_DIR / 'fig2f_slideseq_beads.parquet')
    ss['cell_type'] = ss['cell_type'].replace({'PT-S1':'PT-S1/2','PT-S2':'PT-S1/2'})
    for g in FIG2F_GENES: ss[g+'_n'] = ss[g]/ss['nUMI'].replace(0,np.nan)*1e4
    pct = pd.DataFrame(index=FIG2F_GENES, columns=FIG2F_TYPES, dtype=float)
    avg = pd.DataFrame(index=FIG2F_GENES, columns=FIG2F_TYPES, dtype=float)
    for c in FIG2F_TYPES:
        sub=ss[ss['cell_type']==c]
        for g in FIG2F_GENES: pct.loc[g,c]=(sub[g]>0).mean()*100; avg.loc[g,c]=sub[g+'_n'].mean()
    return pct, _scaled(avg)

def compute_vis():
    lab = pd.read_parquet(PROCESSED_DATA_DIR / 'fig2f_visium_labeled.parquet')
    V = sc.read_h5ad(PROCESSED_DATA_DIR / 'fig2f_visium_raw.h5ad')
    sc.pp.normalize_total(V, target_sum=1e4); sc.pp.log1p(V); V=V[lab.index]
    labv = lab['pred_subclass_l2'].values
    Xg = V[:,FIG2F_GENES].X; Xg = Xg.toarray() if sps.issparse(Xg) else np.asarray(Xg)
    pct = pd.DataFrame(0.0,index=FIG2F_GENES, columns=FIG2F_TYPES)
    avg = pd.DataFrame(0.0,index=FIG2F_GENES, columns=FIG2F_TYPES)
    for c in FIG2F_TYPES:
        m=labv==c
        if m.sum()==0: continue
        pct[c]=(Xg[m]>0).mean(axis=0)*100; avg[c]=np.expm1(Xg[m]).mean(axis=0)
    return pct, _scaled(avg)

print('Computing snCv3...'); snc_pct, snc_sc = compute_snc()
print('Computing Slide-seq...'); ss_pct, ss_sc = compute_ss()
print('Computing Visium...'); vis_pct, vis_sc = compute_vis()
print('Three-platform statistics complete')

CMAP_SNC = LinearSegmentedColormap.from_list('snc',['#E0E0E0','#3B5BA5','#10204A'])
CMAP_SS  = LinearSegmentedColormap.from_list('ss', ['#EDE6DA','#E8923A','#9A4B06'])
CMAP_VIS = LinearSegmentedColormap.from_list('vis',['#E5E0EC','#9B59C6','#4A1B6B'])
PANELS = [('snCv3/scCv3', snc_pct, snc_sc, CMAP_SNC, (25,100)),
          ('Slide-seq2',  ss_pct,  ss_sc,  CMAP_SS,  (20,60)),
          ('Visium',      vis_pct, vis_sc, CMAP_VIS, (25,75))]
SMAX=120; VMAX=2.0; ng,nt=len(FIG2F_GENES),len(FIG2F_TYPES)

def plot_panel(ax, pct, scaled, cmap, title, show_y):
    pct=pct.loc[FIG2F_GENES,FIG2F_TYPES]; scaled=scaled.loc[FIG2F_GENES,FIG2F_TYPES]; norm=Normalize(0,VMAX)
    for gi,g in enumerate(FIG2F_GENES):
        for ti,t in enumerate(FIG2F_TYPES):
            ax.scatter(ti, ng-1-gi, s=pct.loc[g,t]/100*SMAX, color=cmap(norm(np.clip(scaled.loc[g,t],0,VMAX))), edgecolors='none')
    ax.set_xticks(range(nt)); ax.set_xticklabels(FIG2F_TYPES, rotation=90, fontsize=8)
    if show_y: ax.set_yticks(range(ng)); ax.set_yticklabels(FIG2F_GENES[::-1], fontsize=9, style='italic')
    else: ax.set_yticks([])
    ax.set_xlim(-0.6,nt-0.4); ax.set_ylim(-0.6,ng-0.4); ax.set_title(title,fontsize=11)
    ax.set_aspect('equal'); ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_linewidth(0.6)

fig = plt.figure(figsize=(11.5,5.8), facecolor='white')
gs = gridspec.GridSpec(2,3, height_ratios=[1,0.16], hspace=0.02, wspace=0.12)
for i,(title,pct,scaled,cmap,sref) in enumerate(PANELS):
    plot_panel(fig.add_subplot(gs[0,i]), pct, scaled, cmap, title, show_y=(i==0))
for i,(title,pct,scaled,cmap,sref) in enumerate(PANELS):
    leg = gridspec.GridSpecFromSubplotSpec(1,2, subplot_spec=gs[1,i], wspace=0.5)
    axs = fig.add_subplot(leg[0]); axs.set_title('Exp. (%)',fontsize=8, pad=2)
    for j,v in enumerate(sref): axs.scatter(j,0,s=v/100*SMAX,color='#555',edgecolors='none')
    axs.set_xticks(range(len(sref))); axs.set_xticklabels([str(v) for v in sref],fontsize=8)
    axs.set_xlim(-0.7,len(sref)-0.3); axs.set_ylim(-0.5,0.5); axs.set_yticks([]); axs.tick_params(length=0)
    for s in axs.spines.values(): s.set_visible(False)
    axc = fig.add_subplot(leg[1])
    cb = mcolorbar.ColorbarBase(axc, cmap=cmap, norm=Normalize(0,VMAX), orientation='horizontal', ticks=[0,1,2])
    cb.set_label('Ave. Exp.', fontsize=8); cb.ax.tick_params(labelsize=8); axc.set_box_aspect(0.16)

FIG2F_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG2F_OUTPUT, dpi=300, bbox_inches='tight', facecolor='white')
print('Saved', FIG2F_OUTPUT.name)
plt.show(); plt.close(fig)
print('Figure 2f complete')


---
## Module 8 — Figure 2g reproduction

Manuscript Fig. 2g: healthy reference Visium section (cortex Cx above, outer medulla OM below) showing H&E histology and per-spot **transfer scores** / gene expression.

**Six panels (left → right)**: Histology · POD · C-TAL · M-TAL · DTL2 · *SLC12A1*

| Element | Description |
|---------|-------------|
| Sample | `GSM6047774` / `V19S25-016_XY01_18-0006` (Indiana BBCI healthy reference nephrectomy) |
| H&E | GEO supplementary `*.tif.gz` (Keyence high-resolution mosaic; not Space Ranger PNG) |
| Transfer score | snCv3 reference + PCA ingest + kNN soft assignment (approximates Seurat `TransferData`) |
| Medullary ray outline | manual histological polygon; adjustable vertices provided below |

> **Distinction from Module 7**: Module 7 uses `filtered_feature_bc_matrix.h5` for dot plots only; Fig. 2g additionally requires **spatial coordinates, H&E TIFF, and continuous transfer scores**.


### Module 8a — Data validation + H&E download + spatial extraction

1. Confirm Module 7 `*.tar.gz` archives (Space Ranger outs) are present
2. Download corresponding `*.tif.gz` high-resolution H&E (not fetched in Module 7)
3. Extract `spatial/` and expression matrix from tar.gz into a `sc.read_visium`-compatible layout
4. Cache `fig2g_visium_spatial.h5ad` and H&E TIFF


In [ ]:
import kidney_atlas_paths as kap
# Module 8a — data validation + H&E download + spatial extraction
!pip install -q tifffile
import gzip, json, shutil, tarfile, urllib.request
from pathlib import Path

import scanpy as sc

if 'DATA_DIR' not in globals():
    from pathlib import Path
    DRIVE_ROOT = kap.PROJECT_ROOT
    FIGURES_DIR = DRIVE_ROOT / 'figures'
    DATA_DIR = DRIVE_ROOT / 'data'
    PROCESSED_DATA_DIR = DATA_DIR / 'processed'
    CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'

RAW_DATA_DIR = DATA_DIR / 'raw'
VISIUM_DIR = RAW_DATA_DIR / 'visium' / 'GSE183456'
FIG2G_SAMPLE_GSM = 'GSM6047774'
FIG2G_SAMPLE_ID = 'V19S25-016_XY01_18-0006'
FIG2G_TGZ = VISIUM_DIR / f'{FIG2G_SAMPLE_GSM}_{FIG2G_SAMPLE_ID}.tar.gz'
FIG2G_TIF_GZ = VISIUM_DIR / f'{FIG2G_SAMPLE_GSM}_{FIG2G_SAMPLE_ID}.tif.gz'
FIG2G_HE_TIFF = PROCESSED_DATA_DIR / 'fig2g_he_18-0006.tif'
FIG2G_SPATIAL_DIR = PROCESSED_DATA_DIR / 'fig2g_visium_18-0006'
FIG2G_H5AD = PROCESSED_DATA_DIR / 'fig2g_visium_spatial.h5ad'
VISIUM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

def _geo_suppl_url(gsm, fname):
    folder = gsm[:-3] + 'nnn'
    return f'https://ftp.ncbi.nlm.nih.gov/geo/samples/{folder}/{gsm}/suppl/{fname}'

def _download(url, dst):
    if dst.exists() and dst.stat().st_size > 1000:
        print('  Already exists; skipping:', dst.name)
        return
    print('  Downloading', dst.name)
    urllib.request.urlretrieve(url, dst)

if not FIG2G_TGZ.exists():
    _download(_geo_suppl_url(FIG2G_SAMPLE_GSM, FIG2G_TGZ.name), FIG2G_TGZ)
print('tar.gz:', FIG2G_TGZ.exists())

if not FIG2G_TIF_GZ.exists():
    _download(_geo_suppl_url(FIG2G_SAMPLE_GSM, FIG2G_TIF_GZ.name), FIG2G_TIF_GZ)
if not FIG2G_HE_TIFF.exists():
    with gzip.open(FIG2G_TIF_GZ, 'rb') as fin, open(FIG2G_HE_TIFF, 'wb') as fout:
        shutil.copyfileobj(fin, fout)
    print('Decompressed H&E ->', FIG2G_HE_TIFF.name)
else:
    print('H&E cache exists:', FIG2G_HE_TIFF.name)

def _extract_visium_outs():
    if FIG2G_SPATIAL_DIR.exists():
        shutil.rmtree(FIG2G_SPATIAL_DIR)
    FIG2G_SPATIAL_DIR.mkdir(parents=True)
    (FIG2G_SPATIAL_DIR / 'spatial').mkdir(exist_ok=True)
    with tarfile.open(FIG2G_TGZ) as tf:
        names = tf.getnames()
        print('Files in tar:', len(names), '| sample:', names[:8])
        for m in tf.getmembers():
            base = Path(m.name).name
            if base == 'filtered_feature_bc_matrix.h5':
                tf.extract(m, FIG2G_SPATIAL_DIR)
                src = FIG2G_SPATIAL_DIR / m.name
                dst = FIG2G_SPATIAL_DIR / base
                if src != dst:
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(src), str(dst))
            elif base in ('tissue_positions_list.csv', 'scalefactors_json.json'):
                tf.extract(m, FIG2G_SPATIAL_DIR)
                src = FIG2G_SPATIAL_DIR / m.name
                dst = FIG2G_SPATIAL_DIR / 'spatial' / base
                if src != dst:
                    shutil.move(str(src), str(dst))
    print('spatial directory:', sorted(p.name for p in (FIG2G_SPATIAL_DIR/'spatial').glob('*')))

if not FIG2G_H5AD.exists():
    _extract_visium_outs()
    # GEO tar lacks PNG; use expression matrix + coordinates; H&E from separate TIFF
    A = sc.read_visium(FIG2G_SPATIAL_DIR, load_images=False)
    A.var_names_make_unique()
    A.obs['sample'] = FIG2G_SAMPLE_ID
    A.write_h5ad(FIG2G_H5AD)
    print('Saved', FIG2G_H5AD.name, A.shape)
else:
    A = sc.read_h5ad(FIG2G_H5AD)
    print('spatial h5ad cache exists:', A.shape)

print('spatial coordinate columns:', [c for c in A.obs.columns if 'pxl' in c or 'array' in c][:6])
print('Module 8a complete')


### Module 8b — Transfer score computation (snCv3 → Visium)

The manuscript applies Seurat `TransferData` for per-spot `subclass.l2` scores. Python approximation:

1. snCv3 reference (`state == "reference"`; ≤800 cells per class)
2. Shared genes → HVG → PCA → `sc.tl.ingest`
3. kNN `predict_proba` in reference PCA space → per-spot type scores
4. Row-normalize non-zero scores per spot (consistent with relative proportions in the manuscript)

Results cached in `fig2g_transfer_scores.parquet`.


In [ ]:
# Module 8b — compute transfer scores (optimized for 12 GB RAM)
import gc, json, time, urllib.request
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import scipy.sparse as sps
from sklearn.neighbors import NearestNeighbors

RAW_DATA_DIR = DATA_DIR / 'raw'
CELLXGENE_DIR = RAW_DATA_DIR / 'cellxgene'
CELLXGENE_DIR.mkdir(parents=True, exist_ok=True)
SNC_H5AD = CELLXGENE_DIR / 'kidney_integrated_sc_snRNA.h5ad'
SNC_URL = 'https://datasets.cellxgene.cziscience.com/4cd166f1-ef51-4137-869d-0a3688bc2bc8.h5ad'
FIG2G_SCORES = PROCESSED_DATA_DIR / 'fig2g_transfer_scores.parquet'
FIG2G_SCORES_META = PROCESSED_DATA_DIR / 'fig2g_transfer_scores.meta.json'
SCORE_METHOD = 'gaussian_knn_k50_v3_mem12g'
REF_CAP = 200       # 12 GB Colab: ≤200 per class (800 may OOM)
N_HVG = 1000
N_PCS = 30
KNN_K = 50
FIG2G_TYPES = ['POD', 'C-TAL', 'M-TAL', 'DTL2']


def _need_recompute_scores():
    if not FIG2G_SCORES.exists() or not FIG2G_SCORES_META.exists():
        return True
    try:
        return json.loads(FIG2G_SCORES_META.read_text()).get('method') != SCORE_METHOD
    except Exception:
        return True


def _to_dense(X):
    return X.toarray() if sps.issparse(X) else np.asarray(X)


def _scale_with_ref(X, mean, std):
    std = np.where(std < 1e-8, 1.0, std)
    X = _to_dense(X).astype(np.float32, copy=False)
    return np.clip((X - mean) / std, -10, 10)


def _gaussian_knn_scores_targets(ref_pca, ref_labels, query_pca, targets, k=50):
    k = min(k, len(ref_pca))
    nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    nn.fit(ref_pca)
    dist, idx = nn.kneighbors(query_pca)
    sigma = max(float(np.median(dist[:, -1])), 1e-6)
    w = np.exp(-0.5 * (dist / sigma) ** 2)
    w /= w.sum(axis=1, keepdims=True)
    labels = np.asarray(ref_labels, dtype=str)
    neighbor_labels = labels[idx]
    out = np.zeros((query_pca.shape[0], len(targets)), dtype=np.float32)
    for j, ct in enumerate(targets):
        out[:, j] = (w * (neighbor_labels == ct)).sum(axis=1)
    return pd.DataFrame(out, columns=targets)


if not SNC_H5AD.exists():
    print('Downloading snCv3 integrated reference (~3 GB)...')
    urllib.request.urlretrieve(SNC_URL, SNC_H5AD)
    print('Saved', SNC_H5AD)
else:
    print('snCv3 reference exists:', f'{SNC_H5AD.stat().st_size/1e9:.2f}GB')

if not _need_recompute_scores():
    scores = pd.read_parquet(FIG2G_SCORES)
    print('Score cache exists:', scores.shape, f'({SCORE_METHOD})')
else:
    if FIG2G_SCORES.exists():
        print('Stale score cache detected; recomputing with updated method...')
    gc.collect()
    t0 = time.time()

    V_bk = sc.read_h5ad(FIG2G_H5AD, backed='r')
    spot_ids = V_bk.obs_names
    Vgenes = set(map(str, V_bk.var_names))

    A = ad.read_h5ad(SNC_H5AD, backed='r')
    sym = A.var['feature_name'].astype(str).values
    common_cols = np.where(np.isin(sym, list(Vgenes)))[0]
    state = A.obs['state'].astype(str).values
    sub = A.obs['subclass.l2'].astype(str).values
    is_ref = state == 'reference'
    rng = np.random.default_rng(0)
    idx = []
    for c in np.unique(sub[is_ref]):
        pos = np.where((sub == c) & is_ref)[0]
        if len(pos) < 20:
            continue
        idx.extend((pos if len(pos) <= REF_CAP else rng.choice(pos, REF_CAP, replace=False)).tolist())
    idx = np.sort(np.array(idx, dtype=np.int64))
    print(f'Reference subsample {len(idx)} cells, {len(np.unique(sub[idx]))} types')

    ref = A[idx, common_cols].to_memory()
    ref.var_names = sym[common_cols]
    ref.var_names_make_unique()
    del A, state, sub, is_ref, sym, common_cols
    gc.collect()

    sc.pp.normalize_total(ref, target_sum=1e4)
    sc.pp.log1p(ref)
    sc.pp.highly_variable_genes(ref, n_top_genes=N_HVG, subset=True)
    hvg = ref.var_names.tolist()
    print(f'HVG={len(hvg)}, reference matrix {ref.shape}')

    sc.pp.scale(ref, max_value=10)
    sc.pp.pca(ref, n_comps=N_PCS)
    ref_pca = ref.obsm['X_pca'].astype(np.float32)
    ref_labels = ref.obs['subclass.l2'].astype(str).values
    ref_mean = ref.var['mean'].to_numpy(dtype=np.float32)
    ref_std = ref.var['std'].to_numpy(dtype=np.float32)
    PCs = ref.varm['PCs'].astype(np.float32)
    del ref
    gc.collect()

    Vq = V_bk[:, hvg].to_memory()
    del V_bk
    gc.collect()
    sc.pp.normalize_total(Vq, target_sum=1e4)
    sc.pp.log1p(Vq)
    Vq_pca = _scale_with_ref(Vq.X, ref_mean, ref_std) @ PCs
    del Vq, ref_mean, ref_std, PCs
    gc.collect()

    scores = _gaussian_knn_scores_targets(ref_pca, ref_labels, Vq_pca, FIG2G_TYPES, k=KNN_K)
    scores.index = spot_ids
    del ref_pca, ref_labels, Vq_pca, spot_ids
    gc.collect()

    scores.to_parquet(FIG2G_SCORES)
    FIG2G_SCORES_META.write_text(json.dumps({'method': SCORE_METHOD, 'k': KNN_K, 'ref_cap': REF_CAP}))
    print(f'Transfer scores complete {time.time()-t0:.0f}s -> {FIG2G_SCORES.name}')

for ct in FIG2G_TYPES:
    if ct in scores.columns:
        s = scores[ct]
        print(f'  {ct}: max={s.max():.3f}, p95={s.quantile(0.95):.3f}, p99={s.quantile(0.99):.3f}')
print('Module 8b complete')


### Module 8c — Figure 2g rendering

Six horizontally arranged panels: H&E + four cell-type transfer scores + *SLC12A1* expression.

- Colormap: Spectral (consistent with KPMP official Visium scripts)
- Score ranges: POD 0–0.75 · C-TAL/M-TAL 0–0.8 · DTL2 0–0.5 · SLC12A1 1–4
- Field of view: rightmost 40% of tissue (medullary ray region)
- Scale bar: 300 µm (derived from `scalefactors_json.json`)


In [ ]:
# Module 8c — render Figure 2g
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, PowerNorm
from matplotlib.cm import ScalarMappable
from PIL import Image
import scanpy as sc

Image.MAX_IMAGE_PIXELS = None

FIG2G_OUTPUT = FIGURES_DIR / 'figure2g_visium_spatial.png'
FIG2G_HE_PNG = PROCESSED_DATA_DIR / 'fig2g_he_18-0006.png'
FIG2G_PANELS = [
    ('Histology', None, None, None),
    ('POD', 'POD', 0.75, (0, 0.75)),
    ('C-TAL', 'C-TAL', 0.8, (0, 0.8)),
    ('M-TAL', 'M-TAL', 0.8, (0, 0.8)),
    ('DTL2', 'DTL2', 0.5, (0, 0.5)),
    ('SLC12A1', 'SLC12A1', 4.0, (1, 4)),
]
# Manuscript Fig. 2g medullary ray: rightmost 40% of tissue width
FIG2G_CROP_X_FRAC = (0.6, 1.0)  # rightmost 40% of tissue width
FIG2G_CROP_Y_FRAC = (0.0, 1.0)
# Transfer-score colormap gamma<1: stretch mid-low values for manuscript-like contrast
FIG2G_SCORE_GAMMA = 0.40
def _load_he():
    if FIG2G_HE_PNG.exists():
        return np.array(Image.open(FIG2G_HE_PNG))
    he = np.array(Image.open(FIG2G_HE_TIFF))
    Image.fromarray(he).save(FIG2G_HE_PNG)
    print('Cached H&E PNG ->', FIG2G_HE_PNG.name)
    return he

V = sc.read_h5ad(FIG2G_H5AD)
scores = pd.read_parquet(FIG2G_SCORES)
V = V[scores.index].copy()
he = _load_he()
if he.ndim == 3 and he.shape[-1] not in (3, 4):
    he = np.moveaxis(he, 0, -1)

pos_csv = FIG2G_SPATIAL_DIR / 'spatial' / 'tissue_positions_list.csv'
pos = pd.read_csv(pos_csv, header=None, names=['barcode','in_tissue','array_row','array_col','pxl_row_in_fullres','pxl_col_in_fullres'])
pos = pos.set_index('barcode').reindex(V.obs_names)
xs = pos['pxl_col_in_fullres'].astype(float).values
ys = pos['pxl_row_in_fullres'].astype(float).values
keep = pos['in_tissue'].astype(int).values == 1
xs, ys = xs[keep], ys[keep]
V = V[keep].copy()
scores = scores.loc[V.obs_names]

Vn = V.copy()
sc.pp.normalize_total(Vn, target_sum=1e4); sc.pp.log1p(Vn)
slc_x = Vn[:, 'SLC12A1'].X
slc = np.ravel(slc_x.toarray() if hasattr(slc_x, 'toarray') else slc_x).astype(float)

sf_path = FIG2G_SPATIAL_DIR / 'spatial' / 'scalefactors_json.json'
spot_px = json.loads(sf_path.read_text()).get('spot_diameter_fullres', 88)
bar_px = 300.0 / (55.0 / spot_px)
cmap = plt.cm.Spectral_r
x_min, x_max = float(xs.min()), float(xs.max())
y_min, y_max = float(ys.min()), float(ys.max())
tx, ty = x_max - x_min, y_max - y_min
pad_x, pad_y = 120, 200
crop_x0 = int(x_min + tx * FIG2G_CROP_X_FRAC[0]) - pad_x
crop_x1 = int(x_min + tx * FIG2G_CROP_X_FRAC[1]) + pad_x
crop_y0 = int(y_min + ty * FIG2G_CROP_Y_FRAC[0]) - pad_y
crop_y1 = int(y_min + ty * FIG2G_CROP_Y_FRAC[1]) + pad_y
crop_x0, crop_x1 = max(0, crop_x0), min(he.shape[1], crop_x1)
crop_y0, crop_y1 = max(0, crop_y0), min(he.shape[0], crop_y1)
he_crop = he[crop_y0:crop_y1, crop_x0:crop_x1]
print(f'Crop region x=[{crop_x0},{crop_x1}] y=[{crop_y0},{crop_y1}]')
# Expression panels: fade H&E background to emphasize spot colors (manuscript style)
he_expr = np.clip(he_crop.astype(np.float32) * 0.35 + 255 * 0.65, 0, 255).astype(np.uint8)

def _spot_size_pt2(ax, diameter_px):
    """Convert Visium spot diameter (fullres px) to matplotlib scatter s (points²)."""
    ax.figure.canvas.draw()
    dx_pt = ax.transData.transform((diameter_px, 0)) - ax.transData.transform((0, 0))
    return float(abs(dx_pt[0])) ** 2 * 0.92

def _add_scalebar(ax, x0, y_bar):
    """Scale bar + label (label above bar to avoid bottom clipping)."""
    gap = 0.018 * (crop_y1 - crop_y0)
    ax.plot([x0, x0 + bar_px], [y_bar, y_bar], color='k', lw=2, solid_capstyle='butt')
    ax.text(x0 + bar_px / 2, y_bar - gap, '300 µm', ha='center', va='bottom', fontsize=7)

fig = plt.figure(figsize=(14, 4.8), facecolor='white')
gs = gridspec.GridSpec(1, 6, wspace=0.08)
spot_s = None
for i, (title, key, vmax, clim) in enumerate(FIG2G_PANELS):
    ax = fig.add_subplot(gs[0, i])
    bg = he_crop if key is None else he_expr
    ax.imshow(bg, extent=[crop_x0, crop_x1, crop_y1, crop_y0])
    ax.set_xlim(crop_x0, crop_x1); ax.set_ylim(crop_y1, crop_y0)
    if key is None:
        ax.set_title(title, fontsize=10, bbox=dict(boxstyle='round,pad=0.2', fc='#E8E8E8', ec='none'))
        rx = crop_x1 - 0.04 * (crop_x1 - crop_x0)
        ax.text(rx, crop_y0 + 0.10 * (crop_y1 - crop_y0), 'Cx', fontsize=9, fontweight='bold', ha='right')
        ax.text(rx, crop_y1 - 0.10 * (crop_y1 - crop_y0), 'OM', fontsize=9, fontweight='bold', ha='right')
        ax.annotate('', xy=(rx + 40, crop_y1 - 0.12 * (crop_y1 - crop_y0)),
                    xytext=(rx + 40, crop_y0 + 0.12 * (crop_y1 - crop_y0)),
                    arrowprops=dict(arrowstyle='->', color='k', lw=1.0))
    elif key == 'SLC12A1':
        cvals = np.clip(slc, clim[0], clim[1])
        order = np.argsort(cvals)
        if spot_s is None:
            spot_s = _spot_size_pt2(ax, spot_px)
        ax.scatter(xs[order], ys[order], c=cvals[order], s=spot_s, cmap=cmap,
                   norm=Normalize(clim[0], clim[1]), linewidths=0, alpha=0.98,
                   zorder=3, rasterized=True)
        ax.set_title(r'$\mathit{SLC12A1}$', fontsize=10, bbox=dict(boxstyle='round,pad=0.2', fc='#E8E8E8', ec='none'))
    else:
        vals = scores[key].to_numpy(dtype=float) if key in scores.columns else np.zeros(len(V))
        cvals = np.clip(vals, clim[0], clim[1])
        order = np.argsort(cvals)
        if spot_s is None:
            spot_s = _spot_size_pt2(ax, spot_px)
        score_norm = PowerNorm(gamma=FIG2G_SCORE_GAMMA, vmin=clim[0], vmax=clim[1])
        ax.scatter(xs[order], ys[order], c=cvals[order], s=spot_s, cmap=cmap,
                   norm=score_norm, edgecolors='#1a1a1a', linewidths=0.12,
                   alpha=0.98, zorder=3, rasterized=True)
        ax.set_title(key, fontsize=10, bbox=dict(boxstyle='round,pad=0.2', fc='#E8E8E8', ec='none'))
    y_bar = crop_y1 - 0.03 * (crop_y1 - crop_y0)
    _add_scalebar(ax, crop_x1 - bar_px - 0.10 * (crop_x1 - crop_x0), y_bar)
    if key is not None:
        cb_norm = score_norm if key != 'SLC12A1' else Normalize(clim[0], clim[1])
        sm = ScalarMappable(cmap=cmap, norm=cb_norm); sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.02)
        cb.set_ticks([clim[0], clim[1]]); cb.ax.tick_params(labelsize=6)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_linewidth(0.6)

fig.subplots_adjust(bottom=0.12)
fig.text(0.5, 0.03, 'Visium', ha='center', fontsize=11, fontweight='bold')
# Note: do not savefig directly to /content/drive/MyDrive/ (FUSE may not sync to cloud)
# Use Module 2 save_figure_to_drive() for Drive API upload
save_figure_to_drive(fig, FIG2G_OUTPUT.name, dpi=300, pad_inches=0.12)
plt.show(); plt.close(fig)
print('Figure 2g complete')


---
## Module 9 — Figure 3a reproduction

Manuscript Fig. 3a: mean proportion of **altered-state expression signatures** across all Visium spots (22 donors; 146,460 spots).

**Method** (aligned with KPMP [`state_barplot.R`](https://github.com/KPMP/Cell-State-Atlas-2022/blob/develop/SourceByTechnology/Visium/state_barplot.R)):

1. snCv3 full `subclass.l2` → Visium spot **TransferData approximation** (kNN soft assignment)
2. Row-normalize subclass scores per spot; aggregate by `state.l2` into six classes: Ref / Degen / Cyc / Trans / aEpi / aStr
3. Compute group proportions by CKD / AKI / Ref; Fisher's exact test (vs Ref)

> **Memory**: Module 9b processes samples sequentially and caches `fig3a_spots/*.parquet`; suitable for 12 GB Colab.


### Module 9a — Data validation + download + sample grouping

Verify 23 Visium `tar.gz` archives; fetch disease→condition table from GEO; build `subclass.l2` → `state.l2` mapping.


In [ ]:
import kidney_atlas_paths as kap
# Module 9a — data validation + Visium download + sample grouping
import json, re, time, urllib.request
from pathlib import Path
import anndata as ad

if 'DATA_DIR' not in globals():
    DRIVE_ROOT = kap.PROJECT_ROOT
    DATA_DIR = DRIVE_ROOT / 'data'
    PROCESSED_DATA_DIR = DATA_DIR / 'processed'
    FIGURES_DIR = DRIVE_ROOT / 'figures'

RAW_DATA_DIR = DATA_DIR / 'raw'
VISIUM_DIR = RAW_DATA_DIR / 'visium' / 'GSE183456'
VISIUM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

FIG3A_MANIFEST = PROCESSED_DATA_DIR / 'fig3a_visium_manifest.json'
FIG3A_STATE_MAP = PROCESSED_DATA_DIR / 'fig3a_subclass_state_map.csv'
FIG3A_REF_CACHE = PROCESSED_DATA_DIR / 'fig3a_transfer_ref.npz'
FIG3A_SPOTS_DIR = PROCESSED_DATA_DIR / 'fig3a_spots'
FIG3A_SPOTS_DIR.mkdir(exist_ok=True)
FIG3A_BAR_DATA = PROCESSED_DATA_DIR / 'fig3a_barplot_data.csv'
FIG3A_OUTPUT = FIGURES_DIR / 'figure3a_altered_state_barplot.png'

GSM_IDS = [f'GSM{n}' for n in range(6047774, 6047797)]

def _geo_suppl_url(gsm, fname):
    folder = gsm[:-3] + 'nnn'
    return f'https://ftp.ncbi.nlm.nih.gov/geo/samples/{folder}/{gsm}/suppl/{fname}'

def _to_condition(disease: str) -> str:
    d = disease.lower()
    if 'reference' in d:
        return 'Ref'
    if 'aki' in d or 'acute' in d:
        return 'AKI'
    return 'CKD'

def _fetch_geo_manifest():
    rows = []
    for gsm in GSM_IDS:
        url = f'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gsm}&targ=self&form=text&view=brief'
        txt = urllib.request.urlopen(url, timeout=30).read().decode('utf-8', 'ignore')
        title = re.search(r'!Sample_title = (.+)', txt)
        disease = re.search(r'disease: (.+)', txt)
        tgz = re.findall(r'(GSM\d+_[^\s]+\.tar\.gz)', txt)
        rows.append({
            'gsm': gsm,
            'title': title.group(1).strip() if title else gsm,
            'disease': disease.group(1).strip() if disease else '',
            'condition': _to_condition(disease.group(1) if disease else ''),
            'tar_gz': tgz[0] if tgz else '',
        })
        time.sleep(0.15)
    return rows

def _state_l2_label(st_l2: str, l1: str) -> str:
    s = st_l2.lower()
    if 'reference' in s:
        return 'Ref'
    if 'degenerative' in s:
        return 'Degen'
    if 'cycling' in s:
        return 'Cyc'
    if 'transitioning' in s:
        return 'Trans'
    if 'adaptive' in s and 'epi' in s:
        return 'aEpi'
    if 'adaptive' in s and 'str' in s:
        return 'aStr'
    AEPI_L1 = {'POD','PEC','PT','DTL','ATL','TAL','DCT','CNT','PC','IC','PapE'}
    if 'adaptive' in s:
        return 'aEpi' if l1 in AEPI_L1 else 'aStr'
    return 'Ref'

# 1) Visium tar.gz download (reuses Module 7a logic)
if FIG3A_MANIFEST.exists():
    manifest = json.loads(FIG3A_MANIFEST.read_text())
    print('Manifest cache exists:', len(manifest), 'samples')
else:
    manifest = _fetch_geo_manifest()
    FIG3A_MANIFEST.write_text(json.dumps(manifest, indent=2))
    print('Saved manifest ->', FIG3A_MANIFEST.name)

missing = [m for m in manifest if m['tar_gz'] and not (VISIUM_DIR / m['tar_gz']).exists()]
print('Visium tar.gz:', len(list(VISIUM_DIR.glob('*.tar.gz'))), '/', len(manifest), '| pending:', len(missing))
for m in missing:
    dst = VISIUM_DIR / m['tar_gz']
    gsm = m['gsm']
    url = _geo_suppl_url(gsm, m['tar_gz'])
    print('Downloading', m['tar_gz'])
    urllib.request.urlretrieve(url, dst)

from collections import Counter
print('Condition distribution:', dict(Counter(m['condition'] for m in manifest)))

# 2) subclass.l2 → state.l2 mapping (obs['state.l2']; aligned with KPMP state_barplot.R)
SNC_H5AD = RAW_DATA_DIR / 'cellxgene' / 'kidney_integrated_sc_snRNA.h5ad'
import pandas as pd
if FIG3A_STATE_MAP.exists():
    mapdf = pd.read_csv(FIG3A_STATE_MAP)
    if mapdf['state.l2'].nunique() <= 1:
        print('Incorrect mapping detected; rebuilding state table...')
        FIG3A_STATE_MAP.unlink()
        mapdf = None
if not FIG3A_STATE_MAP.exists():
    A = ad.read_h5ad(SNC_H5AD, backed='r')
    obs = A.obs[['subclass.l2', 'state.l2', 'subclass.l1']].astype(str)
    rows = []
    uniq = obs.drop_duplicates(subset=['subclass.l2'])
    for _, row in uniq.iterrows():
        sub, st2, l1 = row['subclass.l2'], row['state.l2'], row['subclass.l1']
        label = _state_l2_label(st2, l1)
        rows.append({'subclass.l2': sub, 'state.l2': label, 'subclass.l1': l1, 'state_raw': st2})
    mapdf = pd.DataFrame(rows).drop_duplicates('subclass.l2')
    mapdf = mapdf.sort_values('subclass.l2').head(74)
    mapdf.to_csv(FIG3A_STATE_MAP, index=False)
    print('State mapping saved:', mapdf['state.l2'].value_counts().to_dict())
else:
    mapdf = pd.read_csv(FIG3A_STATE_MAP)
    print('State mapping cache:', mapdf.shape, mapdf['state.l2'].value_counts().to_dict())

print('Module 9a complete')


### Module 9b — Per-sample transfer + spot state scores

Build reference PCA once (cache `fig3a_transfer_ref.npz`); compute subclass predictions per sample and aggregate into six state scores.


In [ ]:
# Module 9b — per-sample label transfer + spot-level state scores (12 GB RAM friendly)
import gc, json, tarfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import scipy.sparse as sps
from sklearn.neighbors import NearestNeighbors

REF_CAP = 120          # per-subclass.l2 cap (all states, not reference-only)
N_HVG = 1000
N_PCS = 30
KNN_K = 50

manifest = json.loads(FIG3A_MANIFEST.read_text())
mapdf = pd.read_csv(FIG3A_STATE_MAP)
subclasses = mapdf['subclass.l2'].tolist()
sub2state = dict(zip(mapdf['subclass.l2'], mapdf['state.l2']))
states = ['Ref', 'Degen', 'Cyc', 'Trans', 'aEpi', 'aStr']

SNC_H5AD = RAW_DATA_DIR / 'cellxgene' / 'kidney_integrated_sc_snRNA.h5ad'


def _to_dense(X):
    return X.toarray() if sps.issparse(X) else np.asarray(X)


def _scale_with_ref(X, mean, std):
    std = np.where(std < 1e-8, 1.0, std)
    X = _to_dense(X).astype(np.float32, copy=False)
    return np.clip((X - mean) / std, -10, 10)


def _gaussian_knn_proba(ref_pca, ref_labels, query_pca, classes, k=50):
    k = min(k, len(ref_pca))
    nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    nn.fit(ref_pca)
    dist, idx = nn.kneighbors(query_pca)
    sigma = max(float(np.median(dist[:, -1])), 1e-6)
    w = np.exp(-0.5 * (dist / sigma) ** 2)
    w /= w.sum(axis=1, keepdims=True)
    labels = np.asarray(ref_labels, dtype=str)
    nb = labels[idx]
    out = np.zeros((query_pca.shape[0], len(classes)), dtype=np.float32)
    for j, ct in enumerate(classes):
        out[:, j] = (w * (nb == ct)).sum(axis=1)
    return out


def _build_ref_model():
    if FIG3A_REF_CACHE.exists():
        z = np.load(FIG3A_REF_CACHE, allow_pickle=True)
        return z['ref_pca'], z['ref_labels'].astype(str), z['hvg'].astype(str)

    A = ad.read_h5ad(SNC_H5AD, backed='r')
    sym = A.var['feature_name'].astype(str).values
    sub = A.obs['subclass.l2'].astype(str).values
    rng = np.random.default_rng(0)
    idx = []
    for c in subclasses:
        pos = np.where(sub == c)[0]
        if len(pos) == 0:
            continue
        idx.extend((pos if len(pos) <= REF_CAP else rng.choice(pos, REF_CAP, replace=False)).tolist())
    idx = np.sort(np.array(idx, dtype=np.int64))
    print(f'Reference subsample {len(idx)} cells, {len(subclasses)} subclasses')
    ref = A[idx, :].to_memory()
    if 'feature_name' in ref.var.columns:
        ref.var_names = ref.var['feature_name'].astype(str).to_numpy()
    ref.var_names_make_unique()
    del A; gc.collect()

    sc.pp.normalize_total(ref, target_sum=1e4)
    sc.pp.log1p(ref)
    sc.pp.highly_variable_genes(ref, n_top_genes=N_HVG, subset=True)
    hvg = ref.var_names.astype(str).to_numpy()
    sc.pp.scale(ref, max_value=10)
    sc.pp.pca(ref, n_comps=N_PCS)
    ref_pca = ref.obsm['X_pca'].astype(np.float32)
    ref_labels = ref.obs['subclass.l2'].astype(str).to_numpy()
    ref_mean = ref.var['mean'].to_numpy(dtype=np.float32)
    ref_std = ref.var['std'].to_numpy(dtype=np.float32)
    PCs = ref.varm['PCs'].astype(np.float32)
    np.savez_compressed(FIG3A_REF_CACHE, ref_pca=ref_pca, ref_labels=ref_labels, hvg=hvg,
                      ref_mean=ref_mean, ref_std=ref_std, PCs=PCs)
    del ref; gc.collect()
    return ref_pca, ref_labels, hvg


def _extract_h5(tgz_path: Path, work: Path):
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    (work / 'spatial').mkdir(exist_ok=True)
    with tarfile.open(tgz_path) as tf:
        for m in tf.getmembers():
            base = Path(m.name).name
            if base == 'filtered_feature_bc_matrix.h5':
                tf.extract(m, work)
                src = work / m.name
                dst = work / base
                if src != dst:
                    shutil.move(str(src), str(dst))
            elif base in ('tissue_positions_list.csv', 'scalefactors_json.json'):
                tf.extract(m, work)
                src = work / m.name
                dst = work / 'spatial' / base
                if src != dst:
                    shutil.move(str(src), str(dst))


z = np.load(FIG3A_REF_CACHE, allow_pickle=True) if FIG3A_REF_CACHE.exists() else None
if z is None:
    ref_pca, ref_labels, hvg = _build_ref_model()
    z = np.load(FIG3A_REF_CACHE, allow_pickle=True)
ref_mean, ref_std, PCs = z['ref_mean'], z['ref_std'], z['PCs']
ref_pca, ref_labels, hvg = z['ref_pca'], z['ref_labels'].astype(str), z['hvg'].astype(str)

pending = [m for m in manifest if m['tar_gz'] and not (FIG3A_SPOTS_DIR / f"{m['gsm']}.parquet").exists()]
print('Samples pending:', len(pending), '/', len(manifest))

for mi, m in enumerate(pending):
    gsm, tgz_name = m['gsm'], m['tar_gz']
    tgz = VISIUM_DIR / tgz_name
    out_pq = FIG3A_SPOTS_DIR / f'{gsm}.parquet'
    print(f"[{mi+1}/{len(pending)}] {gsm} {m['condition']} {m['title']}")
    work = PROCESSED_DATA_DIR / f'_tmp_visium_{gsm}'
    _extract_h5(tgz, work)
    V = sc.read_visium(work, load_images=False)
    V.var_names_make_unique()
    hv = [g for g in hvg if g in V.var_names]
    Vq = V[:, hv].copy()
    sc.pp.normalize_total(Vq, target_sum=1e4)
    sc.pp.log1p(Vq)
    gi = np.array([list(hvg).index(g) for g in hv])
    Vq_pca = _scale_with_ref(Vq.X, ref_mean[gi], ref_std[gi]) @ PCs[gi, :]
    proba = _gaussian_knn_proba(ref_pca, ref_labels, Vq_pca, subclasses, k=KNN_K)
    proba = proba / np.clip(proba.sum(axis=1, keepdims=True), 1e-9, None)
    df = pd.DataFrame(proba, columns=subclasses, index=Vq.obs_names)
    state_mat = np.zeros((df.shape[0], len(states)), dtype=np.float32)
    for j, sub in enumerate(subclasses):
        st = sub2state.get(sub, 'Ref')
        sj = states.index(st)
        state_mat[:, sj] += df[sub].to_numpy()
    spot_df = pd.DataFrame(state_mat, columns=states, index=df.index)
    spot_df['condition'] = m['condition']
    spot_df['gsm'] = gsm
    spot_df['title'] = m['title']
    spot_df.to_parquet(out_pq)
    shutil.rmtree(work, ignore_errors=True)
    del V, Vq, Vq_pca, proba, df, spot_df; gc.collect()

n_done = len(list(FIG3A_SPOTS_DIR.glob('*.parquet')))
print(f'Module 9b complete: {n_done} sample spot tables cached')


### Module 9c — Figure 3a rendering

Aggregate all spots, apply Fisher's test, render grouped bar plot; upload `figure3a_altered_state_barplot.png`.


In [ ]:
# Module 9c — aggregate proportions + Fisher test + render Figure 3a
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

manifest = json.loads(FIG3A_MANIFEST.read_text())
states = ['Ref', 'Degen', 'Cyc', 'Trans', 'aEpi', 'aStr']
conds = ['CKD', 'AKI', 'Ref']
colors = {'CKD': '#E64B35', 'AKI': '#00A087', 'Ref': '#4A6FA5'}

parts = [pd.read_parquet(FIG3A_SPOTS_DIR / f"{m['gsm']}.parquet") for m in manifest if (FIG3A_SPOTS_DIR / f"{m['gsm']}.parquet").exists()]
spots = pd.concat(parts, axis=0)
print(f'Total spots: {len(spots):,} (manuscript ~146,460)')

# KPMP state_barplot.R: sum state scores by condition, then normalize to proportions
bardf = spots.groupby('condition')[states].sum().reindex(conds)
prop = bardf.div(bardf.sum(axis=1), axis=0)
prop_long = prop.reset_index().melt(id_vars='condition', var_name='State', value_name='Proportion')
prop_long.to_csv(FIG3A_BAR_DATA, index=False)
print(prop.round(3))

# Fisher exact: each state AKI/CKD vs Ref (aligned with KPMP; aggregated scores as 2×2 table)
pvals = pd.DataFrame(index=states, columns=['AKI', 'CKD'], dtype=float)
raw = bardf.copy()
for st in states:
    for cond in ['AKI', 'CKD']:
        a = raw.loc[cond, st]
        b = raw.loc[cond].sum() - a
        c = raw.loc['Ref', st]
        d = raw.loc['Ref'].sum() - c
        _, p = fisher_exact([[a, b], [c, d]])
        pvals.loc[st, cond] = p

def _stars(p):
    if p < 1e-10: return '***'
    if p < 1e-5: return '**'
    if p < 0.01: return '*'
    return ''

fig, ax = plt.subplots(figsize=(5.2, 4.2))
x = np.arange(len(states))
w = 0.26
for i, cond in enumerate(conds):
    vals = prop.loc[cond, states].to_numpy()
    bars = ax.bar(x + (i - 1) * w, vals, width=w, color=colors[cond], label=cond, edgecolor='none')
    if cond != 'Ref':
        for j, (bar, st) in enumerate(zip(bars, states)):
            star = _stars(pvals.loc[st, cond])
            if star:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
                        star, ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(states)
ax.set_ylabel('Mean proportion (all spots)')
ax.set_ylim(0, max(0.75, prop.values.max() * 1.15))
ax.legend(title='Condition', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_figure_to_drive(fig, FIG3A_OUTPUT.name, dpi=300)
plt.show(); plt.close(fig)
print('Figure 3a complete ->', FIG3A_OUTPUT.name)


---
## Module 10 — Figure 3b reproduction

Manuscript **Fig. 3b**: Visium feature plots of **aEpi** cell state on healthy reference and CKD samples (prediction weight; color scale 0–0.8; scale bar 300 µm).

| Sample | GSM | Notes |
|--------|-----|-------|
| Healthy reference | `GSM6047774` / `18-0006` | Same nephrectomy as Fig. 2g |
| CKD | `GSM6047780` / `IU-13437` | Aligned with KPMP `preliminary_plots.R` |

> **Dependencies**: Module 9a path constants + `fig3a_transfer_ref.npz`; **10c** recomputes transfer scores via Seurat (required for Fig. 3b); **10b** provides optional kNN cache.


### Module 10a — Data validation + H&E download

Confirm Visium tar.gz archives, aEpi column in `fig3a_spots`, and download/cache H&E TIFF for the CKD sample.


In [ ]:
import kidney_atlas_paths as kap
# Module 10a — data validation + H&E download
import gzip, json, shutil, tarfile, urllib.request
from pathlib import Path
import pandas as pd

if 'DATA_DIR' not in globals():
    DRIVE_ROOT = kap.PROJECT_ROOT
    DATA_DIR = DRIVE_ROOT / 'data'
    PROCESSED_DATA_DIR = DATA_DIR / 'processed'
    FIGURES_DIR = DRIVE_ROOT / 'figures'

RAW_DATA_DIR = DATA_DIR / 'raw'
VISIUM_DIR = RAW_DATA_DIR / 'visium' / 'GSE183456'

FIG3B_REF_GSM = 'GSM6047774'
FIG3B_REF_ID = 'V19S25-016_XY01_18-0006'
FIG3B_CKD_GSM = 'GSM6047780'
FIG3B_CKD_ID = 'V19S25-017_XY03-13437'

FIG3B_REF_HE = PROCESSED_DATA_DIR / 'fig2g_he_18-0006.tif'
FIG3B_REF_HE_PNG = PROCESSED_DATA_DIR / 'fig2g_he_18-0006.png'
FIG3B_REF_SPATIAL = PROCESSED_DATA_DIR / 'fig2g_visium_18-0006'
FIG3B_CKD_HE = PROCESSED_DATA_DIR / 'fig3b_he_13437.tif'
FIG3B_CKD_HE_PNG = PROCESSED_DATA_DIR / 'fig3b_he_13437.png'
FIG3B_CKD_SPATIAL = PROCESSED_DATA_DIR / 'fig3b_visium_13437'
FIG3B_SUBCLASS_DIR = PROCESSED_DATA_DIR / 'fig3b_subclass_scores'
FIG3B_SUBCLASS_DIR.mkdir(exist_ok=True)
FIG3B_OUTPUT = FIGURES_DIR / 'figure3b_aepi_featureplot.png'

manifest = json.loads((PROCESSED_DATA_DIR / 'fig3a_visium_manifest.json').read_text())
by_gsm = {m['gsm']: m for m in manifest}

def _geo_suppl_url(gsm, fname):
    folder = gsm[:-3] + 'nnn'
    return f'https://ftp.ncbi.nlm.nih.gov/geo/samples/{folder}/{gsm}/suppl/{fname}'

def _download(url, dst):
    if dst.exists():
        return
    print('Downloading', dst.name)
    urllib.request.urlretrieve(url, dst)

for gsm, sid in [(FIG3B_REF_GSM, FIG3B_REF_ID), (FIG3B_CKD_GSM, FIG3B_CKD_ID)]:
    tgz = VISIUM_DIR / f'{gsm}_{sid}.tar.gz'
    if not tgz.exists():
        m = by_gsm[gsm]
        _download(_geo_suppl_url(gsm, m['tar_gz']), tgz)
    print(gsm, 'tar.gz', tgz.exists())

tif_gz = VISIUM_DIR / f'{FIG3B_CKD_GSM}_{FIG3B_CKD_ID}.tif.gz'
if not FIG3B_CKD_HE.exists():
    if not tif_gz.exists():
        _download(_geo_suppl_url(FIG3B_CKD_GSM, tif_gz.name), tif_gz)
    with gzip.open(tif_gz, 'rb') as fi, open(FIG3B_CKD_HE, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
    print('CKD H&E ->', FIG3B_CKD_HE.name)

for gsm in [FIG3B_REF_GSM, FIG3B_CKD_GSM]:
    pq = PROCESSED_DATA_DIR / 'fig3a_spots' / f'{gsm}.parquet'
    df = pd.read_parquet(pq)
    print(gsm, 'spots', len(df), '| aEpi max', round(df['aEpi'].max(), 3))
print('Module 10a complete')


### Module 10b — Two-sample subclass transfer score cache

Compute subclass soft assignment for Ref and CKD (reusing `fig3a_transfer_ref.npz`); cache `fig3b_subclass_scores/*.parquet` for pie-chart panels.


In [ ]:
# Module 10b — subclass-level transfer scores (two samples only)
import gc, json, tarfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc
import scipy.sparse as sps
from sklearn.neighbors import NearestNeighbors

KNN_K = 50
targets = [
    (FIG3B_REF_GSM, FIG3B_REF_ID, FIG3B_REF_SPATIAL),
    (FIG3B_CKD_GSM, FIG3B_CKD_ID, FIG3B_CKD_SPATIAL),
]

mapdf = pd.read_csv(PROCESSED_DATA_DIR / 'fig3a_subclass_state_map.csv')
subclasses = mapdf['subclass.l2'].tolist()

z = np.load(PROCESSED_DATA_DIR / 'fig3a_transfer_ref.npz', allow_pickle=True)
ref_pca = z['ref_pca']
ref_labels = z['ref_labels'].astype(str)
hvg = z['hvg'].astype(str)
ref_mean, ref_std, PCs = z['ref_mean'], z['ref_std'], z['PCs']


def _to_dense(X):
    return X.toarray() if sps.issparse(X) else np.asarray(X)

def _scale_with_ref(X, mean, std):
    std = np.where(std < 1e-8, 1.0, std)
    X = _to_dense(X).astype(np.float32, copy=False)
    return np.clip((X - mean) / std, -10, 10)

def _gaussian_knn_proba(ref_pca, ref_labels, query_pca, classes, k=50):
    k = min(k, len(ref_pca))
    nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    nn.fit(ref_pca)
    dist, idx = nn.kneighbors(query_pca)
    sigma = max(float(np.median(dist[:, -1])), 1e-6)
    w = np.exp(-0.5 * (dist / sigma) ** 2)
    w /= w.sum(axis=1, keepdims=True)
    labels = np.asarray(ref_labels, dtype=str)
    nb = labels[idx]
    out = np.zeros((query_pca.shape[0], len(classes)), dtype=np.float32)
    for j, ct in enumerate(classes):
        out[:, j] = (w * (nb == ct)).sum(axis=1)
    return out

def _extract_visium(tgz_path: Path, work: Path):
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    (work / 'spatial').mkdir(exist_ok=True)
    with tarfile.open(tgz_path) as tf:
        for m in tf.getmembers():
            base = Path(m.name).name
            if base == 'filtered_feature_bc_matrix.h5':
                tf.extract(m, work)
                src = work / m.name
                dst = work / base
                if src != dst:
                    shutil.move(str(src), str(dst))
            elif base in ('tissue_positions_list.csv', 'scalefactors_json.json'):
                tf.extract(m, work)
                src = work / m.name
                dst = work / 'spatial' / base
                if src != dst:
                    shutil.move(str(src), str(dst))

for gsm, sid, work in targets:
    out_pq = FIG3B_SUBCLASS_DIR / f'{gsm}.parquet'
    if out_pq.exists():
        print('Skipping cached', out_pq.name)
        continue
    tgz = VISIUM_DIR / f'{gsm}_{sid}.tar.gz'
    print('Processing', gsm)
    _extract_visium(tgz, work)
    V = sc.read_visium(work, load_images=False)
    V.var_names_make_unique()
    hv = [g for g in hvg if g in V.var_names]
    Vq = V[:, hv].copy()
    sc.pp.normalize_total(Vq, target_sum=1e4)
    sc.pp.log1p(Vq)
    gi = np.array([list(hvg).index(g) for g in hv])
    Vq_pca = _scale_with_ref(Vq.X, ref_mean[gi], ref_std[gi]) @ PCs[gi, :]
    proba = _gaussian_knn_proba(ref_pca, ref_labels, Vq_pca, subclasses, k=KNN_K)
    proba = proba / np.clip(proba.sum(axis=1, keepdims=True), 1e-9, None)
    df = pd.DataFrame(proba, columns=subclasses, index=Vq.obs_names)
    df.to_parquet(out_pq)
    del V, Vq, proba, df; gc.collect()
    print('Saved', out_pq.name)

print('Module 10b complete')


### Module 10c — Seurat TransferData (required for Fig. 3b)

**Reference**: GEO [GSE183277](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE183277) official snCv3 (Counts RDS + Metadata; same source as h5Seurat).

- **All conditions**: stratified `subclass.l2` subsample (`REF_CAP=150` per class), aligned with full KPMP KBR (not Ref-only)
- Full 8 GB `LoadH5Seurat` may OOM on standard Colab RAM; Counts RDS path is more stable (high-RAM environments may use h5Seurat)

Background **23-sample** TransferData ~2–5 h → enables **Fig. 3b** upon completion.

> **Alternative**: if KPMP `all_merged_subclass.l2_norazor.RDS` is available, use **Module 10d** to skip this step.


In [ ]:
import kidney_atlas_paths as kap
# Module 10c — Seurat TransferData (GSE183277 full snCv3 reference → fig3b_seurat_scores)
# Data: GEO GSE183277 Counts RDS + Metadata (same source as official h5Seurat; avoids 8 GB LoadH5Seurat OOM)
# Reference: stratified subclass.l2 subsample across all conditions (REF_CAP per class), aligned with full KPMP KBR (not Ref-only)
import gc, json, shutil, subprocess, tarfile, textwrap
from pathlib import Path
from datetime import datetime
import pandas as pd

DRIVE_ROOT = kap.PROJECT_ROOT
DATA_DIR = DRIVE_ROOT / 'data'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
RAW_DATA_DIR = DATA_DIR / 'raw'
VISIUM_DIR = RAW_DATA_DIR / 'visium' / 'GSE183456'
FIG3A_MANIFEST = PROCESSED_DATA_DIR / 'fig3a_visium_manifest.json'

GEO_DIR = RAW_DATA_DIR / 'geo' / 'GSE183277'
GEO_DIR.mkdir(parents=True, exist_ok=True)
SNC_COUNTS_GZ = GEO_DIR / 'GSE183277_Kidney_Healthy-Injury_Cell_Atlas_snCv3_Counts_03282022.RDS.gz'
SNC_COUNTS_RDS = GEO_DIR / 'GSE183277_Kidney_Healthy-Injury_Cell_Atlas_snCv3_Counts_03282022.RDS'
SNC_META_GZ = GEO_DIR / 'GSE183277_Kidney_Healthy-Injury_Cell_Atlas_snCv3_Metadata_03282022.txt.gz'
SNC_META_TXT = GEO_DIR / 'GSE183277_Kidney_Healthy-Injury_Cell_Atlas_snCv3_Metadata_03282022.txt'
GEO_URLS = {
    SNC_COUNTS_GZ.name: 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE183277&format=file&file=GSE183277_Kidney_Healthy-Injury_Cell_Atlas_snCv3_Counts_03282022.RDS.gz',
    SNC_META_GZ.name: 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE183277&format=file&file=GSE183277_Kidney_Healthy-Injury_Cell_Atlas_snCv3_Metadata_03282022.txt.gz',
}

FIG3B_SEURAT_DIR = PROCESSED_DATA_DIR / 'fig3b_seurat_scores'
FIG3B_SEURAT_DIR.mkdir(exist_ok=True)
FIG3B_SEURAT_META = FIG3B_SEURAT_DIR / '_meta.json'
FIG3B_R_DIR = PROCESSED_DATA_DIR / 'fig3b_r_transfer'
FIG3B_R_DIR.mkdir(exist_ok=True)
FIG3C_SCORES_DIR = PROCESSED_DATA_DIR / 'fig3c_subclass_scores'
REF_CAP = 150
SCT_N_GENES = 3000
REF_FILTER = f'GSE183277_full_snCv3_cap{REF_CAP}'
TRANSFER_LEVEL = 'subclass.l2'
R_SCRIPT = PROCESSED_DATA_DIR / 'fig3b_seurat_transfer.R'
STATUS = PROCESSED_DATA_DIR / 'fig3b_seurat_status.txt'
LOG = PROCESSED_DATA_DIR / 'fig3b_seurat_r.log'
PIDF = PROCESSED_DATA_DIR / 'fig3b_seurat_r.pid'
LAUNCHER = PROCESSED_DATA_DIR / 'fig3b_seurat_launch.sh'

manifest = json.loads(FIG3A_MANIFEST.read_text())
mapdf = pd.read_csv(PROCESSED_DATA_DIR / 'fig3a_subclass_state_map.csv')
subclasses = mapdf['subclass.l2'].tolist()
OUT_SPEC = {
    'reference': 'GSE183277_counts_rds',
    'ref_cap': REF_CAP,
    'sct_n_genes': SCT_N_GENES,
    'ref_filter': REF_FILTER,
    'transfer_level': TRANSFER_LEVEL,
    'subclasses': len(subclasses),
    'n_manifest': len(manifest),
}


def _log(msg):
    print(msg)
    with open(STATUS, 'a') as f:
        f.write(msg + '\n')


def _job_alive():
    if not PIDF.exists():
        return False
    pid = PIDF.read_text().strip()
    return subprocess.run(['bash', '-lc', f'kill -0 {pid} 2>/dev/null'], capture_output=True).returncode == 0


def _ensure_geo_reference():
    for fname, url in GEO_URLS.items():
        dest = GEO_DIR / fname
        min_size = 100_000_000 if fname.endswith('.RDS.gz') else 1_000_000
        if dest.exists() and dest.stat().st_size >= min_size:
            print('Exists:', dest.name, f'({dest.stat().st_size/1e6:.1f} MB)')
            continue
        print(f'Downloading {fname}…')
        subprocess.run(['wget', '-c', '-O', str(dest), url], check=True)
    if SNC_META_GZ.exists() and not SNC_META_TXT.exists():
        subprocess.run(['gzip', '-dk', str(SNC_META_GZ)], check=True)
    if SNC_COUNTS_GZ.exists() and not SNC_COUNTS_RDS.exists():
        print('Decompressing Counts RDS.gz…')
        subprocess.run(['gzip', '-dk', str(SNC_COUNTS_GZ)], check=True)


def _invalidate_old_outputs():
    cur = None
    if FIG3B_SEURAT_META.exists():
        try:
            cur = json.loads(FIG3B_SEURAT_META.read_text())
        except json.JSONDecodeError:
            cur = None
    if cur == OUT_SPEC:
        print('Spec unchanged; keeping existing CSV files')
        return False
    print('Reference set changed → clearing stale fig3b/fig3c outputs')
    for p in FIG3B_SEURAT_DIR.glob('GSM*.*'):
        p.unlink()
    ref_sct = FIG3B_R_DIR / 'ref_sct.rds'
    if ref_sct.exists():
        ref_sct.unlink()
    if FIG3C_SCORES_DIR.exists():
        for p in FIG3C_SCORES_DIR.glob('GSM*.parquet'):
            p.unlink()
    FIG3B_SEURAT_META.write_text(json.dumps(OUT_SPEC, indent=2))
    return True


def _extract_visium(tgz_path, work):
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    (work / 'spatial').mkdir(exist_ok=True)
    with tarfile.open(tgz_path) as tf:
        for m in tf.getmembers():
            base = Path(m.name).name
            if base == 'filtered_feature_bc_matrix.h5':
                tf.extract(m, work)
                src = work / m.name
                dst = work / base
                if src != dst:
                    shutil.move(str(src), str(dst))
            elif base in ('tissue_positions_list.csv', 'scalefactors_json.json', 'tissue_lowres_image.png'):
                tf.extract(m, work)
                src = work / m.name
                dst = work / 'spatial' / base
                if src != dst:
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(src), str(dst))


SCT_ARGS = (
    f"variable.features.n = {SCT_N_GENES}, return.only.var.genes = TRUE, "
    "conserve.memory = TRUE, verbose = TRUE, vst.flavor = 'v2'"
)

_ensure_geo_reference()
ref_changed = _invalidate_old_outputs()
if _job_alive():
    if not ref_changed:
        print('Background job already running PID=', PIDF.read_text().strip())
        raise SystemExit(0)
    subprocess.run(['bash', '-lc', f'kill {PIDF.read_text().strip()} 2>/dev/null || true'], check=False)

pd.Series(subclasses).to_csv(FIG3B_R_DIR / 'subclasses.csv', index=False, header=False)

done = {p.stem for p in FIG3B_SEURAT_DIR.glob('GSM*.csv')}
todo = [m for m in manifest if m.get('tar_gz') and m['gsm'] not in done]
print(f'Seurat {len(done)}/{len(manifest)} | pending {len(todo)}')

r_build_ref = textwrap.dedent(f"""
  build_ref <- function() {{
    counts_rds <- '{SNC_COUNTS_RDS.as_posix()}'
    meta_txt <- '{SNC_META_TXT.as_posix()}'
    ref_rds <- paste0(rdir, '/ref_sct.rds')
    if (file.exists(ref_rds)) {{
      cat('load cached ref SCT\\n'); ref <- readRDS(ref_rds); DefaultAssay(ref) <- 'SCT'; return(ref)
    }}
    cat('load GSE183277 counts RDS...\\n'); flush.console()
    counts <- readRDS(counts_rds)
    meta <- read.delim(meta_txt, sep='\\t', header=TRUE, row.names=1, check.names=FALSE)
    common <- intersect(colnames(counts), rownames(meta))
    counts <- counts[, common, drop=FALSE]
    meta <- meta[common, , drop=FALSE]
    stopifnot('subclass.l2' %in% colnames(meta))
    set.seed(0)
    keep <- c()
    for (cl in unique(meta$subclass.l2)) {{
      idx <- which(meta$subclass.l2 == cl)
      n <- min({REF_CAP}, length(idx))
      keep <- c(keep, sample(idx, n))
    }}
    keep <- sort(unique(keep))
    cat('subsample', length(keep), 'cells from', length(common), '(cap={REF_CAP}/subclass)\\n'); flush.console()
    ref <- CreateSeuratObject(counts = counts[, keep, drop=FALSE], meta.data = meta[keep, , drop=FALSE], min.cells = 3, min.features = 0)
    rm(counts, meta); gc()
    ref <- SCTransform(ref, {SCT_ARGS})
    DefaultAssay(ref) <- 'SCT'
    saveRDS(ref, ref_rds)
    ref
  }}
""")

r_header = textwrap.dedent(f"""
  suppressPackageStartupMessages({{ library(Seurat); library(Matrix) }})
  options(future.globals.maxSize = 8000 * 1024^2)
  rdir <- '{FIG3B_R_DIR.as_posix()}'
  outdir <- '{FIG3B_SEURAT_DIR.as_posix()}'
  subclasses <- scan(paste0(rdir, '/subclasses.csv'), what='character', quiet=TRUE)
  {r_build_ref}
  ref <- build_ref(); gc()
  run_one <- function(gsm, visium_dir, out_csv) {{
    if (file.exists(out_csv)) {{ cat('skip', gsm, '\\n'); return(invisible(NULL)) }}
    cat('TransferData', gsm, '\\n'); flush.console()
    spatial <- Load10X_Spatial(data.dir = visium_dir, assay = 'Spatial', slice = 'slice1')
    spatial <- SCTransform(spatial, assay = 'Spatial', {SCT_ARGS})
    spatial <- RunPCA(spatial, assay = 'SCT', verbose = FALSE)
    anchors <- FindTransferAnchors(reference = ref, query = spatial, normalization.method = 'SCT', recompute.residuals = TRUE)
    pred <- TransferData(anchorset = anchors, refdata = ref$subclass.l2, prediction.assay = TRUE,
                         weight.reduction = spatial[['pca']], dims = 1:30)
    mat <- as.matrix(GetAssayData(pred, layer = 'data'))
    mat <- mat[intersect(rownames(mat), subclasses), , drop=FALSE]
    write.csv(t(mat), out_csv, quote=FALSE)
    rm(spatial, anchors, pred, mat); gc()
  }}
""")

lines = [r_header]
for m in manifest:
    if not m.get('tar_gz'):
        continue
    gsm = m['gsm']
    out_csv = FIG3B_SEURAT_DIR / f'{gsm}.csv'
    work = PROCESSED_DATA_DIR / f'_tmp_seurat_{gsm}'
    tgz = VISIUM_DIR / m['tar_gz']
    if not out_csv.exists():
        if not tgz.exists():
            print('WARN', tgz.name)
            continue
        if not (work / 'filtered_feature_bc_matrix.h5').exists():
            _extract_visium(tgz, work)
    lines.append(f"run_one('{gsm}', '{work.as_posix()}', '{out_csv.as_posix()}')\n")
R_SCRIPT.write_text(''.join(lines))

meta_json = json.dumps(OUT_SPEC)
LAUNCHER.write_text(f"""#!/bin/bash
set -euo pipefail
LOG="{LOG.as_posix()}"
STATUS="{STATUS.as_posix()}"
RSCRIPT="{R_SCRIPT.as_posix()}"
echo "===== $(date -Iseconds) GSE183277 counts-RDS launch =====" >> "$LOG"
if ! command -v Rscript >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev
fi
Rscript -e "pkgs=c('Seurat','Matrix','sp'); miss=pkgs[!sapply(pkgs,requireNamespace,quietly=TRUE)]; if(length(miss)) install.packages(miss, repos='https://cloud.r-project.org')" >> "$LOG" 2>&1
Rscript -e "if (!requireNamespace('BiocManager', quietly=TRUE)) install.packages('BiocManager', repos='https://cloud.r-project.org'); if (!requireNamespace('glmGamPoi', quietly=TRUE)) BiocManager::install('glmGamPoi', ask=FALSE, update=FALSE)" >> "$LOG" 2>&1
echo "$(date -Iseconds) start GSE183277 TransferData cap{REF_CAP}" >> "$STATUS"
Rscript "$RSCRIPT" >> "$LOG" 2>&1
python3 -c "import json,pandas as pd; from pathlib import Path; seurat=Path('{FIG3B_SEURAT_DIR.as_posix()}'); meta=Path('{FIG3B_SEURAT_META.as_posix()}');\\nfor csv in sorted(seurat.glob('GSM*.csv')):\\n df=pd.read_csv(csv,index_col=0); df.index.name='barcode'; df.to_parquet(csv.with_suffix('.parquet')); print(csv.name, df.shape)\\nmeta.write_text({meta_json!r})"
echo "$(date -Iseconds) 10c done → 10e (Fig.3b)" >> "$STATUS"
echo "===== $(date -Iseconds) done =====" >> "$LOG"
""")
LAUNCHER.chmod(0o755)

_log(f'Background start GSE183277 (cap={REF_CAP}) @ {datetime.now().isoformat(timespec="seconds")}')
subprocess.run(['bash', '-lc', f'nohup "{LAUNCHER.as_posix()}" >> "{LOG.as_posix()}" 2>&1 & echo $! > "{PIDF.as_posix()}"'], check=True)
print('PID', PIDF.read_text().strip(), '| pending', len(todo), '| log', LOG)
print('Note: full LoadH5Seurat requires high RAM; using GSE183277 Counts RDS + stratified subsample')


In [ ]:
import kidney_atlas_paths as kap
# Module 10c — Seurat TransferData (R) → fig3b_seurat_scores
# Note: full R pipeline ~30–60 min; this cell launches a background job and returns immediately (avoids MCP/Colab cell timeout)
import gc, shutil, subprocess, tarfile, textwrap
from pathlib import Path
from datetime import datetime
import anndata as ad
import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sps


if 'FIG3B_REF_GSM' not in globals():
    DRIVE_ROOT = kap.PROJECT_ROOT
    DATA_DIR = DRIVE_ROOT / 'data'
    PROCESSED_DATA_DIR = DATA_DIR / 'processed'
    RAW_DATA_DIR = DATA_DIR / 'raw'
    VISIUM_DIR = RAW_DATA_DIR / 'visium' / 'GSE183456'
    FIG3B_REF_GSM = 'GSM6047774'
    FIG3B_REF_ID = 'V19S25-016_XY01_18-0006'
    FIG3B_CKD_GSM = 'GSM6047780'
    FIG3B_CKD_ID = 'V19S25-017_XY03-13437'
    FIG3B_REF_SPATIAL = PROCESSED_DATA_DIR / 'fig2g_visium_18-0006'
    FIG3B_CKD_SPATIAL = PROCESSED_DATA_DIR / 'fig3b_visium_13437'

FIG3B_SEURAT_DIR = PROCESSED_DATA_DIR / 'fig3b_seurat_scores'
FIG3B_SEURAT_DIR.mkdir(exist_ok=True)
FIG3B_REF_SUB = PROCESSED_DATA_DIR / 'fig3b_ref_subsample.h5ad'
FIG3B_R_DIR = PROCESSED_DATA_DIR / 'fig3b_r_transfer'
SNC_H5AD = RAW_DATA_DIR / 'cellxgene' / 'kidney_integrated_sc_snRNA.h5ad'
REF_CAP = 120
R_SCRIPT = PROCESSED_DATA_DIR / 'fig3b_seurat_transfer.R'
STATUS = PROCESSED_DATA_DIR / 'fig3b_seurat_status.txt'
LOG = PROCESSED_DATA_DIR / 'fig3b_seurat_r.log'
PIDF = PROCESSED_DATA_DIR / 'fig3b_seurat_r.pid'
LAUNCHER = PROCESSED_DATA_DIR / 'fig3b_seurat_launch.sh'

targets = [
    (FIG3B_REF_GSM, FIG3B_REF_ID, FIG3B_REF_SPATIAL),
    (FIG3B_CKD_GSM, FIG3B_CKD_ID, FIG3B_CKD_SPATIAL),
]
mapdf = pd.read_csv(PROCESSED_DATA_DIR / 'fig3a_subclass_state_map.csv')
subclasses = mapdf['subclass.l2'].tolist()


def _log(msg):
    print(msg)
    with open(STATUS, 'a') as f:
        f.write(msg + '\n')


def _extract_visium(tgz_path: Path, work: Path):
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    (work / 'spatial').mkdir(exist_ok=True)
    with tarfile.open(tgz_path) as tf:
        for m in tf.getmembers():
            base = Path(m.name).name
            if base == 'filtered_feature_bc_matrix.h5':
                tf.extract(m, work)
                src = work / m.name
                dst = work / base
                if src != dst:
                    shutil.move(str(src), str(dst))
            elif base in ('tissue_positions_list.csv', 'scalefactors_json.json', 'tissue_lowres_image.png'):
                tf.extract(m, work)
                src = work / m.name
                dst = work / 'spatial' / base
                if src != dst:
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(src), str(dst))


def _build_ref_subsample():
    if FIG3B_REF_SUB.exists():
        print('Reference subset exists:', FIG3B_REF_SUB.name)
        return
    print('Building snCv3 subset reference (≤', REF_CAP, ' per subclass)…')
    A = ad.read_h5ad(SNC_H5AD, backed='r')
    sub = A.obs['subclass.l2'].astype(str).values
    rng = np.random.default_rng(0)
    idx = []
    for c in subclasses:
        pos = np.where(sub == c)[0]
        if len(pos) == 0:
            continue
        idx.extend((pos if len(pos) <= REF_CAP else rng.choice(pos, REF_CAP, replace=False)).tolist())
    idx = np.sort(np.array(idx, dtype=np.int64))
    ref = A[idx, :].to_memory()
    if 'feature_name' in ref.var.columns:
        ref.var_names = ref.var['feature_name'].astype(str).to_numpy()
    ref.var_names_make_unique()
    ref.write_h5ad(FIG3B_REF_SUB)
    del A, ref
    gc.collect()


def _export_ref_mtx(rdir: Path):
    rdir.mkdir(parents=True, exist_ok=True)
    ref = ad.read_h5ad(FIG3B_REF_SUB)
    if 'counts' in ref.layers:
        X = ref.layers['counts']
    elif ref.raw is not None:
        X = ref.raw.X
    else:
        X = ref.X
    X = X.tocsr() if sps.issparse(X) else sps.csr_matrix(X)
    scipy.io.mmwrite(rdir / 'ref.mtx', X.T.tocsc())
    pd.DataFrame({'gene': ref.var_names}).to_csv(rdir / 'ref_genes.tsv', sep='\t', index=False, header=False)
    pd.DataFrame({'barcode': ref.obs_names}).to_csv(rdir / 'ref_barcodes.tsv', sep='\t', index=False, header=False)
    ref.obs[['subclass.l2']].astype(str).to_csv(rdir / 'ref_meta.csv')
    del ref
    gc.collect()


def _job_alive():
    if not PIDF.exists():
        return False
    pid = PIDF.read_text().strip()
    r = subprocess.run(['bash', '-lc', f'kill -0 {pid} 2>/dev/null'], capture_output=True)
    return r.returncode == 0


if _job_alive():
    print('Background job already running PID=', PIDF.read_text().strip())
    print('Keep Colab runtime connected; periodically run the 10d-status cell to monitor', LOG.name)
    raise SystemExit(0)

_build_ref_subsample()
_export_ref_mtx(FIG3B_R_DIR)
pd.Series(subclasses).to_csv(FIG3B_R_DIR / 'subclasses.csv', index=False, header=False)

r_header = textwrap.dedent(f"""
  suppressPackageStartupMessages({{
    library(Seurat)
    library(Matrix)
  }})
  rdir <- '{FIG3B_R_DIR.as_posix()}'
  outdir <- '{FIG3B_SEURAT_DIR.as_posix()}'
  subclasses <- scan('{FIG3B_R_DIR.as_posix()}/subclasses.csv', what='character', quiet=TRUE)

  counts <- ReadMtx(mtx = paste0(rdir, '/ref.mtx'), features = paste0(rdir, '/ref_genes.tsv'),
                    cells = paste0(rdir, '/ref_barcodes.tsv'), feature.column = 1, cell.column = 1)
  meta <- read.csv(paste0(rdir, '/ref_meta.csv'), row.names = 1)
  ref <- CreateSeuratObject(counts = counts, meta.data = meta, min.cells = 0, min.features = 0)
  rm(counts, meta); gc()
  cat('SCTransform reference...\\n'); flush.console()
  ref <- SCTransform(ref, verbose = TRUE, vst.flavor = 'v2')
  DefaultAssay(ref) <- 'SCT'

  run_one <- function(gsm, visium_dir, out_csv) {{
    if (file.exists(out_csv)) {{ cat('skip', gsm, '\\n'); return(invisible(NULL)) }}
    cat('TransferData', gsm, '\\n'); flush.console()
    spatial <- Load10X_Spatial(data.dir = visium_dir, assay = 'Spatial', slice = 'slice1')
    spatial <- SCTransform(spatial, assay = 'Spatial', verbose = TRUE, vst.flavor = 'v2')
    spatial <- RunPCA(spatial, assay = 'SCT', verbose = FALSE)
    anchors <- FindTransferAnchors(reference = ref, query = spatial, normalization.method = 'SCT', recompute.residuals = TRUE)
    pred <- TransferData(anchorset = anchors, refdata = ref$subclass.l2, prediction.assay = TRUE,
                         weight.reduction = spatial[['pca']], dims = 1:30)
    mat <- as.matrix(GetAssayData(pred, slot = 'data'))
    mat <- mat[intersect(rownames(mat), subclasses), , drop=FALSE]
    write.csv(t(mat), out_csv, quote=FALSE)
    rm(spatial, anchors, pred, mat); gc()
  }}
""")

lines = [r_header]
for gsm, sid, _ in targets:
    out_csv = FIG3B_SEURAT_DIR / f'{gsm}.csv'
    work = PROCESSED_DATA_DIR / f'_tmp_seurat_{gsm}'
    if not out_csv.exists():
        tgz = VISIUM_DIR / f'{gsm}_{sid}.tar.gz'
        _extract_visium(tgz, work)
    lines.append(f"run_one('{gsm}', '{work.as_posix()}', '{out_csv.as_posix()}')\n")
R_SCRIPT.write_text(''.join(lines))

LAUNCHER.write_text(f"""#!/bin/bash
set -euo pipefail
LOG="{LOG.as_posix()}"
STATUS="{STATUS.as_posix()}"
RSCRIPT="{R_SCRIPT.as_posix()}"
SEURAT="{FIG3B_SEURAT_DIR.as_posix()}"
echo "===== $(date -Iseconds) fig3b seurat launch =====" >> "$LOG"
if ! command -v Rscript >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev
fi
Rscript -e "pkgs=c('Seurat','Matrix','sp'); miss=pkgs[!sapply(pkgs,requireNamespace,quietly=TRUE)]; if(length(miss)) install.packages(miss, repos='https://cloud.r-project.org')" >> "$LOG" 2>&1
Rscript -e "if (!requireNamespace('BiocManager', quietly=TRUE)) install.packages('BiocManager', repos='https://cloud.r-project.org'); if (!requireNamespace('glmGamPoi', quietly=TRUE)) BiocManager::install('glmGamPoi', ask=FALSE, update=FALSE)" >> "$LOG" 2>&1
echo "$(date -Iseconds) start TransferData" >> "$STATUS"
Rscript "$RSCRIPT" >> "$LOG" 2>&1
python3 - <<'PY'
from pathlib import Path
import pandas as pd
seurat = Path("{FIG3B_SEURAT_DIR.as_posix()}")
for csv in seurat.glob("*.csv"):
    pq = csv.with_suffix(".parquet")
    if not pq.exists():
        df = pd.read_csv(csv, index_col=0)
        df.index.name = "barcode"
        df.to_parquet(pq)
        print("parquet", pq.name, df.shape)
PY
echo "$(date -Iseconds) 10c complete → rerun 10e" >> "$STATUS"
echo "===== $(date -Iseconds) done =====" >> "$LOG"
""")
LAUNCHER.chmod(0o755)

if LOG.exists():
    LOG.unlink()
_log(f'Background start Seurat TransferData @ {datetime.now().isoformat(timespec="seconds")}')
subprocess.run(['bash', '-lc', f'nohup "{LAUNCHER.as_posix()}" >> "{LOG.as_posix()}" 2>&1 & echo $! > "{PIDF.as_posix()}"'], check=True)
print('PID', PIDF.read_text().strip())
print('Log', LOG)
print('⚠️ Keep Colab runtime connected for 30–60 min; use 10d-status cell below, then rerun 10e when complete')


### Module 10d — KPMP merged Visium (skip 10c)

**Recommended path**: use KPMP official pipeline output, aligned with [`all_apithelial_colocalization.R`](https://github.com/KPMP/Cell-State-Atlas-2022/blob/develop/SourceByTechnology/Visium/all_apithelial_colocalization.R).

**Data access** (KPMP DUA required; not available as a single GEO file):
1. [KPMP Atlas Repository](https://atlas.kpmp.org/repository/) → DOI [`10.48698/3z31-8924`](https://doi.org/10.48698/3z31-8924) → filter **Visium / Aggregated**
2. Or email **info@kpmp.org** for controlled-access processed objects
3. Place `all_merged_subclass.l2_norazor.RDS` under `data/raw/kpmp/`

**This module**: first 74 `subclass.l2` columns from `predictions` assay → column normalization → split by `orig.ident` → `fig3c_subclass_scores/` (`score_source=kpmp_merged`). Enables **Module 10e** for Fig. 3b (or optional subclass score cache).


In [ ]:
import kidney_atlas_paths as kap
# Module 10d — extract subclass.l2 scores from KPMP merged RDS → fig3c_subclass_scores
import gc, json, re, subprocess, textwrap
from pathlib import Path
import pandas as pd

if kap.PROJECT_ROOT.exists():
    DRIVE_ROOT = kap.PROJECT_ROOT
else:
    DRIVE_ROOT = Path('.').resolve()

DATA_DIR = DRIVE_ROOT / 'data'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
RAW_DATA_DIR = DATA_DIR / 'raw'
FIG3A_MANIFEST = PROCESSED_DATA_DIR / 'fig3a_visium_manifest.json'
FIG3A_STATE_MAP = PROCESSED_DATA_DIR / 'fig3a_subclass_state_map.csv'

KPMP_DIR = RAW_DATA_DIR / 'kpmp'
KPMP_DIR.mkdir(parents=True, exist_ok=True)
MERGED_RDS = KPMP_DIR / 'all_merged_subclass.l2_norazor.RDS'
FIG3C_SCORES_DIR = PROCESSED_DATA_DIR / 'fig3c_subclass_scores'
FIG3C_SCORES_DIR.mkdir(exist_ok=True)
EXTRACT_DIR = PROCESSED_DATA_DIR / 'fig3d_kpmp_extract'
EXTRACT_DIR.mkdir(exist_ok=True)
R_SCRIPT = PROCESSED_DATA_DIR / 'fig3d_kpmp_extract.R'
META_OUT = EXTRACT_DIR / '_meta.json'

manifest = json.loads(FIG3A_MANIFEST.read_text())
mapdf = pd.read_csv(FIG3A_STATE_MAP)
subclasses = mapdf['subclass.l2'].tolist()
mapdf[['subclass.l2']].to_csv(EXTRACT_DIR / 'subclasses.csv', index=False, header=False)

# tar.gz name → KPMP orig.ident (e.g. V19S25-016_XY01_18-0006)
sample_rows = []
for m in manifest:
    sid = ''
    if m.get('tar_gz'):
        sid = re.sub(r'^GSM\d+_', '', m['tar_gz']).replace('.tar.gz', '')
    sample_rows.append({'gsm': m['gsm'], 'sample_id': sid, 'condition': m['condition']})
sample_map = pd.DataFrame(sample_rows)
sample_map.to_csv(EXTRACT_DIR / 'sample_map.csv', index=False)

if not MERGED_RDS.exists():
    print('❌ merged RDS not found:')
    print('  ', MERGED_RDS)
    print('Download all_merged_subclass.l2_norazor.RDS from KPMP Repository (DOI 10.48698/3z31-8924)')
    print('Or contact info@kpmp.org for DUA-controlled processed Visium objects.')
    raise SystemExit(1)

print('merged RDS:', MERGED_RDS, f'({MERGED_RDS.stat().st_size/1e9:.2f} GB)')

SUBCLASS_CSV = (EXTRACT_DIR / 'subclasses.csv').as_posix()
SAMPLE_MAP_CSV = (EXTRACT_DIR / 'sample_map.csv').as_posix()

r_code = textwrap.dedent(f"""
  suppressPackageStartupMessages(library(Seurat))
  merged <- readRDS('{MERGED_RDS.as_posix()}')
  if (!'predictions' %in% Assays(merged)) stop('Missing predictions assay')
  mat <- as.matrix(GetAssayData(merged, assay = 'predictions', layer = 'data'))
  subclasses <- scan('{SUBCLASS_CSV}', what = 'character', quiet = TRUE)
  mat <- mat[intersect(rownames(mat), subclasses), , drop = FALSE]
  cs <- colSums(mat)
  mat <- mat[, cs > 0, drop = FALSE]
  mat <- t(t(mat) / colSums(mat))
  smap <- read.csv('{SAMPLE_MAP_CSV}', stringsAsFactors = FALSE)
  meta <- merged@meta.data
  if (!'orig.ident' %in% colnames(meta)) stop('meta.data missing orig.ident')
  outdir <- '{EXTRACT_DIR.as_posix()}'
  for (i in seq_len(nrow(smap))) {{
    gsm <- smap$gsm[i]
    sid <- smap$sample_id[i]
    if (!nzchar(sid)) next
    cells <- rownames(meta)[meta$orig.ident == sid]
    cells <- intersect(cells, colnames(mat))
    if (!length(cells)) {{
      cat('WARN no spots:', gsm, sid, '\\n')
      next
    }}
    sub <- t(mat[, cells, drop = FALSE])
    write.csv(sub, file.path(outdir, paste0(gsm, '.csv')), quote = FALSE)
    cat('OK', gsm, sid, nrow(sub), 'spots\\n')
  }}
""")
R_SCRIPT.write_text(r_code)

subprocess.run(['Rscript', '-e',
    "pkgs=c('Seurat'); miss=pkgs[!sapply(pkgs,requireNamespace,quietly=TRUE)]; "
    "if(length(miss)) install.packages(miss, repos='https://cloud.r-project.org')"],
    check=False)
proc = subprocess.run(['Rscript', str(R_SCRIPT)], capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError('R extraction failed')

n_ok = 0
n_spots = 0
for m in manifest:
    gsm = m['gsm']
    csv = EXTRACT_DIR / f'{gsm}.csv'
    if not csv.exists():
        continue
    df = pd.read_csv(csv, index_col=0)
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    for sub in subclasses:
        if sub not in df.columns:
            df[sub] = 0.0
    df = df[subclasses].astype('float32')
    df['condition'] = m['condition']
    df['gsm'] = gsm
    df['score_source'] = 'kpmp_merged'
    df.to_parquet(FIG3C_SCORES_DIR / f'{gsm}.parquet')
    n_ok += 1
    n_spots += len(df)

meta = {
    'source': 'kpmp_merged',
    'merged_rds': str(MERGED_RDS),
    'n_samples': n_ok,
    'n_spots_total': n_spots,
    'method': 'predictions assay col-normalize (all_apithelial_colocalization.R)',
}
META_OUT.write_text(json.dumps(meta, indent=2))
(PROCESSED_DATA_DIR / 'fig3c_meta.json').write_text(json.dumps({
    'n_spots_total': n_spots,
    'n_samples': n_ok,
    'score_sources': {'kpmp_merged': n_ok},
    'prefer': 'kpmp_merged (10d)',
}, indent=2))
print(f'Module 10d complete: {n_ok}/{len(manifest)} samples, {n_spots:,} spots → fig3c_subclass_scores/')
print('Next: run Module 10e (Fig. 3b)')


### Post-10d status check

Optional: `fig3c_subclass_scores/` from KPMP merged RDS is auxiliary cache; **the reproduction endpoint of this notebook is Fig. 3b (Module 10e)**.


In [ ]:
# Post-10d — verify kpmp_merged scores
from pathlib import Path

FIG3C = PROCESSED_DATA_DIR / 'fig3c_subclass_scores'
n = len(list(FIG3C.glob('*.parquet'))) if FIG3C.exists() else 0
print(f'fig3c_subclass_scores: {n} samples')
if n:
    print('kpmp_merged scores ready; proceed to Module 10e for Fig. 3b')
else:
    print('Module 10e can use fig3b_seurat_scores from 10c without running 10d')


### Module 10d-status — background job monitoring

After running **10c** above, execute the code cell below every 10–15 min to verify R is still running and whether `fig3b_seurat_scores/*.parquet` has been generated.

Run **10e** for main plotting upon completion.


In [ ]:
# Module 10d-status — background job progress (lightweight; re-runnable)
from pathlib import Path
from datetime import datetime
import subprocess

import kidney_atlas_paths as kap
kap.ensure_importable()
PROCESSED = kap.PROCESSED_DATA_DIR
SEURAT = PROCESSED / 'fig3b_seurat_scores'
LOG = PROCESSED / 'fig3b_seurat_r.log'
PIDF = PROCESSED / 'fig3b_seurat_r.pid'
STATUS = PROCESSED / 'fig3b_seurat_status.txt'

def chk(p):
    if not p.exists():
        return 'MISSING'
    st = p.stat()
    return f'OK {st.st_size}B @ {datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds")}'

print('=== Output ===')
for name in ['GSM6047774.csv', 'GSM6047774.parquet', 'GSM6047780.csv', 'GSM6047780.parquet']:
    print(name, chk(SEURAT / name))

print('\n=== Process ===')
if PIDF.exists():
    pid = PIDF.read_text().strip()
    alive = subprocess.run(['bash', '-lc', f'kill -0 {pid} 2>/dev/null && echo alive || echo dead'], capture_output=True, text=True).stdout.strip()
    print(f'PID {pid} -> {alive}')
else:
    print('No PID file')

ps = subprocess.run(['bash', '-lc', 'ps aux | grep -E "fig3b_seurat_launch|fig3b_seurat_transfer" | grep -v grep || echo none'], capture_output=True, text=True)
print('ps:', ps.stdout.strip())

print('\n=== log tail ===')
if LOG.exists():
    lines = LOG.read_text(errors='replace').splitlines()
    print(f'({len(lines)} lines)')
    print('\n'.join(lines[-20:]))

if STATUS.exists():
    print('\n=== status.txt tail ===')
    print(STATUS.read_text()[-600:])


### Post-10c status check

Module 10c outputs `fig3b_seurat_scores/` (for Fig. 3b). Run **Module 10e** upon completion.


In [ ]:
# Post-10c — check Seurat transfer progress
from pathlib import Path

FIG3B = PROCESSED_DATA_DIR / 'fig3b_seurat_scores'
csvs = list(FIG3B.glob('*.csv')) if FIG3B.exists() else []
pqs = list(FIG3B.glob('*.parquet')) if FIG3B.exists() else []
print(f'fig3b_seurat_scores: {len(csvs)} CSV, {len(pqs)} parquet / 23 samples')
if len(csvs) >= 23 or len(pqs) >= 23:
    print('10c complete → run Module 10e for Fig. 3b')
else:
    print('10c still running or not started')


### Module 10e — Figure 3b rendering

Read aEpi from kNN `fig3b_subclass_scores` (Healthy: q0.73; CKD: spatial smoothing it=3); diagnostics printed/saved to `fig3b_aepi_diagnostics.txt` at runtime.


In [ ]:
# Module 10e — Figure 3b main plot
# Healthy: q0.73; CKD: spatial smoothing it=3 (~65% spots colored)
import json, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display, Image as IPyImage
from scipy.spatial import cKDTree

Image.MAX_IMAGE_PIXELS = None

if 'PROCESSED_DATA_DIR' not in globals():
    import kidney_atlas_paths as kap
    kap.ensure_importable()
    kap.bind_notebook_globals(globals())
FIG3B_REF_GSM = 'GSM6047774'
FIG3B_REF_HE_PNG = PROCESSED_DATA_DIR / 'fig2g_he_18-0006.png'
FIG3B_REF_SPATIAL = PROCESSED_DATA_DIR / 'fig2g_visium_18-0006'
FIG3B_REF_HE = PROCESSED_DATA_DIR / 'fig2g_he_18-0006.tif'
FIG3B_CKD_GSM = 'GSM6047780'
FIG3B_CKD_HE_PNG = PROCESSED_DATA_DIR / 'fig3b_he_13437.png'
FIG3B_CKD_SPATIAL = PROCESSED_DATA_DIR / 'fig3b_visium_13437'
FIG3B_CKD_HE = PROCESSED_DATA_DIR / 'fig3b_he_13437.tif'
FIG3B_SUBCLASS_DIR = PROCESSED_DATA_DIR / 'fig3b_subclass_scores'
FIG3B_OUTPUT = FIGURES_DIR / 'figure3b_aepi_featureplot.png'

if 'save_figure_to_drive' not in globals():
    def save_figure_to_drive(fig, filename, dpi=300, pad_inches=0.05, **kwargs):
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', pad_inches=pad_inches, **kwargs)
        out = FIGURES_DIR / filename
        out.write_bytes(buf.getvalue())
        print('Saved', out)
        return out

FIG3B_AEPI_VMAX = 0.25
FIG3B_HEALTHY_Q = 0.73
FIG3B_CKD_Q = 0.00
FIG3B_CKD_SMOOTH_K = 10
FIG3B_CKD_SMOOTH_ITERS = 3
FIG3B_SCALE_UM = 300
FIG3B_DIAG = PROCESSED_DATA_DIR / 'fig3b_aepi_diagnostics.txt'

mapdf = pd.read_csv(PROCESSED_DATA_DIR / 'fig3a_subclass_state_map.csv')
AEPI_SUBS = mapdf.loc[mapdf['state.l2'] == 'aEpi', 'subclass.l2'].tolist()
BREWER = ['#9E0142','#D53E4F','#F46D43','#FDAE61','#FEE08B','#FFFFBF','#E6F598','#ABDDA4','#66C2A5','#3288BD','#5E4FA2']
FIG3B_CMAP = LinearSegmentedColormap.from_list('Spectral_r', BREWER[::-1], N=256)
FIG3B_NORM = Normalize(vmin=0, vmax=FIG3B_AEPI_VMAX, clip=True)
PANELS = [
    ('Healthy reference', FIG3B_REF_GSM, FIG3B_REF_HE_PNG, FIG3B_REF_SPATIAL, FIG3B_REF_HE),
    ('CKD', FIG3B_CKD_GSM, FIG3B_CKD_HE_PNG, FIG3B_CKD_SPATIAL, FIG3B_CKD_HE),
]

def _knn_aepi(gsm):
    sub = pd.read_parquet(FIG3B_SUBCLASS_DIR / f'{gsm}.parquet')
    cols = [c for c in AEPI_SUBS if c in sub.columns]
    rs = sub.sum(axis=1).replace(0, np.nan)
    return (sub[cols].sum(axis=1) / rs).fillna(0).to_numpy(float), sub.index

def _xy(sd, bc):
    pos = pd.read_csv(sd/'spatial/tissue_positions_list.csv', header=None,
        names=['barcode','in_tissue','ar','ac','pxl_row','pxl_col'])
    pos = pos.set_index('barcode').reindex(bc)
    k = pos['in_tissue'].astype(int).values == 1
    return pos['pxl_col'].astype(float).values[k], pos['pxl_row'].astype(float).values[k], k

def _smooth_spatial(v, xs, ys, k=10, iters=1):
    tree = cKDTree(np.c_[xs, ys])
    _, idx = tree.query(np.c_[xs, ys], k=k)
    out = v.astype(float).copy()
    for _ in range(iters):
        out = np.mean(out[idx], axis=1)
    return out

def _display(raw, title, xs, ys):
    raw = np.clip(raw, 0, None)
    if title == 'Healthy reference':
        base = float(np.quantile(raw, FIG3B_HEALTHY_Q))
        disp = np.clip(raw - base, 0, None)
    else:
        sm = _smooth_spatial(raw, xs, ys, FIG3B_CKD_SMOOTH_K, FIG3B_CKD_SMOOTH_ITERS)
        base = float(np.quantile(sm, FIG3B_CKD_Q))
        disp = np.clip(sm - base, 0, None)
    mx = float(disp.max())
    if mx > 0:
        disp = disp / mx * FIG3B_AEPI_VMAX
    return disp, base

def _spot_s(ax, dpx):
    ax.figure.canvas.draw()
    dx = ax.transData.transform((dpx,0)) - ax.transData.transform((0,0))
    return float(abs(dx[0]))**2 * 1.05

def _he(png, tif, sd):
    lr = sd/'spatial/tissue_lowres_image.png'
    if lr.exists(): return np.array(Image.open(lr)), 'lowres'
    if png.exists(): return np.array(Image.open(png)), 'fullres'
    he = np.array(Image.open(tif)); Image.fromarray(he).save(png); return he, 'fullres'

diag = ['Figure 3b diagnostics', f'Healthy q{FIG3B_HEALTHY_Q}; CKD smooth it={FIG3B_CKD_SMOOTH_ITERS}, q{FIG3B_CKD_Q}; vmax={FIG3B_AEPI_VMAX}']
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.8), facecolor='white')
roi_store = {}

for ax, (title, gsm, he_png, sd, he_tif) in zip(axes, PANELS):
    raw, bc = _knn_aepi(gsm)
    xs, ys, k = _xy(sd, bc)
    raw = raw[k]
    disp, base = _display(raw, title, xs, ys)
    diag += [f'=== {title} (base={base:.4f}) ===',
             f'raw med={np.median(raw):.4f} max={raw.max():.4f}',
             f'plot max={disp.max():.4f} colored={(disp>0.02).mean():.1%}']

    he, hk = _he(he_png, he_tif, sd)
    if he.ndim==3 and he.shape[-1] not in (3,4): he = np.moveaxis(he,0,-1)
    sf = json.loads((sd/'spatial/scalefactors_json.json').read_text())
    spx = sf.get('spot_diameter_fullres', 88)
    if hk=='lowres':
        sc = sf.get('tissue_hires_scalef',1)/max(sf.get('tissue_lowres_scalef',0.05),1e-6)
        xs, ys = xs*sc, ys*sc; spx *= sc
    bar = FIG3B_SCALE_UM / (55/spx)
    pad = 0.04*max(xs.max()-xs.min(), ys.max()-ys.min())
    cx0,cx1 = xs.min()-pad, xs.max()+pad
    cy0,cy1 = ys.min()-pad, ys.max()+pad
    bg = np.clip(he.astype(np.float32)*0.55+255*0.45, 0, 255).astype(np.uint8)
    ax.imshow(bg, extent=[0,he.shape[1],he.shape[0],0], zorder=0)
    ax.set_xlim(cx0,cx1); ax.set_ylim(cy1,cy0)
    o = np.argsort(disp)
    ax.scatter(xs[o], ys[o], c=disp[o], s=_spot_s(ax,spx), cmap=FIG3B_CMAP, norm=FIG3B_NORM, linewidths=0, zorder=3, rasterized=True)
    ax.set_title(title, fontweight='bold')
    ax.plot([cx1-bar-0.06*(cx1-cx0), cx1-0.06*(cx1-cx0)], [cy1-0.035*(cy1-cy0)]*2, 'k', lw=2.2)
    if title=='CKD':
        sp=np.sort(ys); split=sp[np.argmax(np.diff(sp))]; mtop=ys<=min(split,np.median(ys))
        box=(xs[mtop].min()-pad, ys[mtop].min()-pad, xs[mtop].max()+pad, ys[mtop].max()+pad)
        ax.add_patch(Rectangle((box[0],box[1]), box[2]-box[0], box[3]-box[1], fill=False, edgecolor='k', lw=1.5))
        roi_store[gsm]=box
    ax.set_xticks([]); ax.set_yticks([])

FIG3B_DIAG.write_text('\n'.join(diag)); print('\n'.join(diag))
sm = ScalarMappable(cmap=FIG3B_CMAP, norm=FIG3B_NORM); sm.set_array([])
fig.colorbar(sm, cax=fig.add_axes([0.72,0.06,0.22,0.025]), orientation='horizontal', label='Prediction weight').set_ticks([0,0.125,0.25])
if roi_store: (PROCESSED_DATA_DIR/'fig3b_ckd_roi.json').write_text(json.dumps({k:list(v) for k,v in roi_store.items()}))
plt.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.12, wspace=0.06)
out_png = save_figure_to_drive(fig, FIG3B_OUTPUT.name, dpi=300, pad_inches=0.05)
plt.close(fig)
display(IPyImage(filename=str(out_png)))
print('Figure 3b ->', out_png)


---
## Scope of reproduction

This notebook **terminates at Figure 3b** (Module 10e).

**Included**: Modules 0–10 (Fig. 1, 2a–2g, 3a, 3b).

**Excluded from maintenance** (removed from notebook):
- Fig. 3c / Fig. 3d (colocalization, etc.)
- Fig. 2d (Visium altered-state dot plot)
- Fig. 4 and beyond

For Fig. 3c / 2d, refer to KPMP [Cell-State-Atlas-2022](https://github.com/KPMP/Cell-State-Atlas-2022) official R scripts.
